In [2]:
# ============================================================
# Project root & path handling
# ------------------------------------------------------------
# Default: working directory
# ============================================================

from pathlib import Path
import os

# Determine project root
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", ".")).expanduser().resolve()

# Convenience function for building repo-relative paths
def p(rel_path):
    """
    Build an absolute path from a path relative to the project root.
    """
    return PROJECT_ROOT / rel_path

print("PROJECT_ROOT set to:", PROJECT_ROOT)

PROJECT_ROOT set to: /net/bbi/vol1/home/mtejura/IGVF-cvfg-pillar-project


In [3]:
# ============================================================
# Create temporary and output directories 
# ============================================================

DATA_DIR = p("data")
TMP_DIR = p("tmp")
OUT_DIR = p("outputs")

TMP_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)


In [163]:
# -----------------------------------------------------------------------------
# Core data harmonization functions
#
# Defines the main helper used to standardize variant annotations across
# heterogeneous supplemental datasets (HGVS.p / HGVS.c, amino acid fields,
# transcript-level information)
# -----------------------------------------------------------------------------


import pandas as pd
import numpy as np
import re
from datetime import datetime


#Data harmonization function to help harmonize variant nomeclature from supplements
def data_harmonization(dc):
    
    aa_dict = {'A': 'Ala','R': 'Arg','N': 'Asn','D': 'Asp','C': 'Cys','E': 'Glu','Q': 'Gln','G': 'Gly','H': 'His',
           'I': 'Ile','L': 'Leu','K': 'Lys','M': 'Met','F': 'Phe','P': 'Pro','S': 'Ser','T': 'Thr','W': 'Trp',
           'Y': 'Tyr','V': 'Val', '=': '=','*': '*', "~": 'del', '-':'del','STOP': '*','Del':'del', "X": 'del'}

    reversed_aa_dict = {v: k for k, v in aa_dict.items()}

    dataset = dc['Dataset']
    
    x = dc['x']
    
    y = dc['y']
    
    df1 = pd.read_excel(x, header = y)
    
    #create an empty dataframe with column names as follows

    pillar_data = pd.DataFrame(columns=["Dataset", "Gene", "HGNC_id","Chrom","hg19_pos","hg38_start","hg38_end","ref_allele",
                                    "alt_allele","auth_transcript_id","transcript_pos","transcript_ref","transcript_alt",
                                   "aa_pos","aa_ref","aa_alt","hgvs_c","hgvs_p","consequence","auth_reported_score",
                                   "auth_reported_rep_score","auth_reported_func_class","splice_measure","gnomad_MAF",
                                    "clinvar_sig","clinvar_star","clinvar_date_last_reviewed","nucleotide_or_aa"])
    
    # Use df1 only to determine the number of rows to create for pillar_data
    num_rows = len(df1)
    
    pillar_data = pd.DataFrame(index=range(num_rows), columns=pillar_data.columns)

    for key, value in dc.items():
        if key in pillar_data.columns:
            if pd.api.types.is_scalar(value):
                pillar_data[key] = value
            else:
                # Handle different data structures properly
                if isinstance(value, np.ndarray):  
                    pillar_data[key] = value  
                elif isinstance(value, pd.Series):  
                    pillar_data[key] = value  
                elif isinstance(value, pd.DataFrame):  
                    if value.shape[1] == 1: 
                        pillar_data[key] = value.iloc[:, 0]
                    else:
                        raise ValueError(f"Expected 1D data for {key}, but got shape {value.shape}")
                else:
                    pillar_data[key] = np.asarray(value)  

                    
    #convert syntax like M1L or p.M1L to hgvs format with three letter amino acid representations
    
    def convert_hgvs_p(hgvs_p):
        if pd.isna(hgvs_p):
            return np.nan
        try:
            # Strip 'p.' if present
            hgvs_p_clean = hgvs_p[2:] if hgvs_p.startswith('p.') else hgvs_p
    
            # Regex pattern: ref_aa (1 letter), position (digits), alt_aa (1 letter or '=')
            match = re.match(r'^([A-Z]|\*)(\d+)([A-Z*=\-~X]|Ter|STOP)$', hgvs_p_clean)
            if not match:
                # If does not match pattern, return original
                return hgvs_p
    
            ref_a, pos, alt_a = match.groups()
            
            # For synonymous, alt = ref (or alt = '=')
            if alt_a == '=':
                alt_a = ref_a
    
            return f"p.{aa_dict[ref_a]}{pos}{aa_dict[alt_a]}"
        except KeyError as e:
            print(f"KeyError converting {dataset}-{hgvs_p}: {e}")
            return hgvs_p

    # Apply
    if dc['hgvs_p_conversion'] == 'Yes':
        pillar_data['hgvs_p'] = pillar_data['hgvs_p'].apply(convert_hgvs_p)
          

    def construct_hgvs_p_from_aa(row):
        try:
            # Handle missing ref/alt cleanly
            if pd.isna(row['aa_ref']) and pd.isna(row['aa_alt']):
                return np.nan
            
            ref_aa = row['aa_ref']
            alt_aa = row['aa_alt']
            pos = row['aa_pos']
            
            # Handle missing aa_pos
            if pd.isna(pos):
                return np.nan
            
            # Construct using aa_dict mapping
            return f"p.{aa_dict[ref_aa]}{pos}{aa_dict[alt_aa]}"
        
        except KeyError as e:
            print(f"KeyError for row {dataset}-{row.name} with aa_ref={row.get('aa_ref')}, aa_alt={row.get('aa_alt')}, aa_pos={row.get('aa_pos')}: {e}")
            # Fallback: return current hgvs_p if it exists
            return row.get('hgvs_p', np.nan)

     
    if dc['hgvs_from_aa'] == 'Yes':
        pillar_data['hgvs_p'] = pillar_data.apply(construct_hgvs_p_from_aa, axis=1)

    

    def extract_aa_from_hgvs(row):
        hgvs_p_pattern = re.compile(r'^p\.([A-Z][a-z]{2}|\*|del|Del)(\d+)([A-Z][a-z]{2}|=|\*|Ter|del|Del)$')
        
        hgvs_p = row['hgvs_p']
        if pd.isna(hgvs_p):
            return pd.Series({'aa_ref': np.nan, 'aa_pos': np.nan, 'aa_alt': np.nan})
        try:
            match = hgvs_p_pattern.match(hgvs_p)
            if not match:
                # If pattern doesn't match, return NaNs for safety
                print(f"No match for row {dataset}-{row.name} with value {hgvs_p}")
                return pd.Series({'aa_ref': np.nan, 'aa_pos': np.nan, 'aa_alt': np.nan})
    
            ref_aa_3, pos, alt_aa_3 = match.groups()
            
            # Map ref_aa from 3-letter to 1-letter
            ref_aa = reversed_aa_dict.get(ref_aa_3, np.nan)
            
            # Handle special cases
            if alt_aa_3 in ['=', 'Ter', '*']:
                if alt_aa_3 == '=':
                    alt_aa = ref_aa
                else:
                    alt_aa = '*'
            else:
                alt_aa = reversed_aa_dict.get(alt_aa_3, np.nan)
    
            return pd.Series({'aa_ref': ref_aa, 'aa_pos': pos, 'aa_alt': alt_aa})
    
        except Exception as e:
            print(f"Error for {dataset}-row {row.name} with value {hgvs_p}: {e}")
            return pd.Series({'aa_ref': np.nan, 'aa_pos': np.nan, 'aa_alt': np.nan})
    
    # Apply
    if dc['aa_from_hgvs'] == 'Yes':
        pillar_data[['aa_ref', 'aa_pos', 'aa_alt']] = pillar_data.apply(extract_aa_from_hgvs, axis=1)
    
    #if hgvs_c is provided, use that information to populate transcript information

    def parse_hgvs_c(row):
        hgvs_c = row['hgvs_c']
        
        output = {
            'transcript_pos': np.nan,
            'transcript_ref': np.nan,
            'transcript_alt': np.nan
        }
        
        if pd.isna(hgvs_c):
            return pd.Series(output)
    
        try:
            # Substitution: c.123A>G, c.*37T>G, c.123+1G>A
            sub_match = re.match(r'c\.([\*\-\d_+\d]+)([ACGT])>([ACGT])$', hgvs_c)
            if sub_match:
                output['transcript_pos'] = sub_match.group(1)
                output['transcript_ref'] = sub_match.group(2)
                output['transcript_alt'] = sub_match.group(3)
                return pd.Series(output)
    
            # Delins: c.123_125delinsG
            delins_match = re.match(r'c\.([\*\d_\+\-]+)delins([ACGT]+)$', hgvs_c)
            if delins_match:
                output['transcript_pos'] = delins_match.group(1)
                output['transcript_ref'] = 'del'
                output['transcript_alt'] = delins_match.group(2)
                return pd.Series(output)
    
            # Deletion: c.123_125del or c.123del or c.123_125delAG
            del_match = re.match(r'c\.([\*\d_\+\-]+)del([ACGT]*)$', hgvs_c)
            if del_match:
                output['transcript_pos'] = del_match.group(1)
                output['transcript_ref'] = del_match.group(2) if del_match.group(2) else 'del'
                output['transcript_alt'] = ''
                return pd.Series(output)
    
            # Insertion: c.123_124insG
            ins_match = re.match(r'c\.([\*\d_\+\-]+)ins([ACGT]+)$', hgvs_c)
            if ins_match:
                output['transcript_pos'] = ins_match.group(1)
                output['transcript_ref'] = ''
                output['transcript_alt'] = ins_match.group(2)
                return pd.Series(output)
    
            # Duplication: c.942dupG or c.995_998dupAGGC
            dup_match = re.match(r'c\.([\*\d_\+\-]+)dup([ACGT]*)$', hgvs_c)
            if dup_match:
                output['transcript_pos'] = dup_match.group(1)
                output['transcript_ref'] = ''
                output['transcript_alt'] = 'dup' + dup_match.group(2)
                return pd.Series(output)
    
            # If no pattern is matched, throw out warning
            print(f"[WARN] Unparsed HGVS_c: {hgvs_c}")
            return pd.Series(output)
    
        except Exception as e:
            print(f"[ERROR] Parsing {hgvs_c} failed: {e}")
            return pd.Series(output)

    # Apply
    if dc.get('transcript_from_hgvs_c') == 'Yes':
        pillar_data[['transcript_pos', 'transcript_ref', 'transcript_alt']] = (
            pillar_data.apply(parse_hgvs_c, axis=1)
        )

    return pillar_data

In [164]:
# Apply data_harmonization across datasets and concatenate results

def data_harmonization_loop(dc_list):
    
    combined_dataframes = []
    
    
    for dc in dc_list:
            harmonized_df = data_harmonization(dc)
        
        
            if harmonized_df is not None and not harmonized_df.empty:
                combined_dataframes.append(harmonized_df) 
    
    
    combined_df = pd.concat(combined_dataframes, ignore_index=True) if combined_dataframes else pd.DataFrame()
    
    return combined_df

In [165]:
#create dataset dictionaries to pull relevant information from supplement files 

df1 = pd.read_excel(DATA_DIR/"Variant_score_files/BRCA1_Findlay_2018.xlsx", header = 2)
dc_1 = {"x": DATA_DIR/"Variant_score_files/BRCA1_Findlay_2018.xlsx", 
               "y": 2,
               "Dataset" :'BRCA1_Findlay_2018', 
               "Gene" : df1['gene'], 
               "HGNC_id" : 1100, 
               "Chrom" : df1['chromosome'], 
               "hg19_pos" : df1['position (hg19)'], 
               "hg38_start" : np.nan,
               "ref_allele": df1['reference'], 
               "alt_allele": df1['alt'], 
               "auth_transcript_id" : df1['transcript_ID'], 
               "transcript_pos" : df1['transcript_position'],
               "transcript_ref" : df1['transcript_ref'], 
               "transcript_alt" : df1['transcript_alt'],
               "aa_pos": df1['aa_pos'],
               "aa_ref": df1['aa_ref'],
               "aa_alt": df1['aa_alt'],
               "hgvs_c": df1['transcript_variant'],
               "hgvs_p" : df1['protein_variant'],
               "consequence": df1['consequence'],
               "auth_reported_score": df1['function.score.mean'],
               "auth_reported_rep_score": df1[['function.score.r1', 'function.score.r2']].fillna('').astype(str).agg(';'.join, axis=1).to_numpy(),
               "auth_reported_func_class": df1['func.class'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#one amino acid change from reference sequence; amino acid 1613 in reference changed from S-->G in experiment
df2 = pd.read_excel(DATA_DIR/"Variant_score_files/BRCA1_Adamovich_2022_HDR.xlsx", header = 0)
dc_2 = {"x": DATA_DIR/"Variant_score_files/BRCA1_Adamovich_2022_HDR.xlsx", 
               "y": 0,
               "Dataset" :'BRCA1_Adamovich_2022_HDR', 
               "Gene" : "BRCA1", 
               "HGNC_id" : 1100, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df2['variantID'],
               "consequence": np.nan,
               "auth_reported_score": df2['FS_average_BRCA1_siRNA'],
               "auth_reported_rep_score": df2[['FS1_BRCA1_siRNA', 'FS2_BRCA1_siRNA','FS3_BRCA1_siRNA','FS4_BRCA1_siRNA']].fillna('').astype(str).agg(';'.join, axis=1).to_numpy(),
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#one amino acid change from reference sequence; amino acid 1613 in reference changed from S-->G in experiment
df3 = pd.read_excel(DATA_DIR/"Variant_score_files/BRCA1_Adamovich_2022_Cisplatin_Resistance.xlsx", header = 0)
dc_3 = {"x": DATA_DIR/"Variant_score_files/BRCA1_Adamovich_2022_Cisplatin_Resistance.xlsx", 
               "y": 0,
               "Dataset" :'BRCA1_Adamovich_2022_Cisplatin_Resistance', 
               "Gene" : "BRCA1", 
               "HGNC_id" : 1100, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df3['variantID'],
               "consequence": np.nan,
               "auth_reported_score": df3['FS_average_BRCA1_siRNA'],
               "auth_reported_rep_score": df3[['FS1_BRCA1_siRNA', 'FS2_BRCA1_siRNA','FS3_BRCA1_siRNA','FS4_BRCA1_siRNA']].fillna('').astype(str).agg(';'.join, axis=1).to_numpy(),
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#p.Val2687A changed to p.Val2687Ala. Mistake in file from authors
df4 = pd.read_excel(DATA_DIR/"Variant_score_files/BRCA2_Hu_2024.xlsx", header = 2)
dc_4 = {"x": DATA_DIR/"Variant_score_files/BRCA2_Hu_2024.xlsx", 
               "y": 2,
               "Dataset" :'BRCA2_Hu_2024', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.3", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df4['coding nucleotide change'],
               "hgvs_p" : df4['Protein change'],
               "consequence": np.nan,
               "auth_reported_score": df4['HDR score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df4['HDR function'],
               "auth_reported_normal_min":2.5,
               "auth_reported_normal_max":np.nan,
               "auth_reported_abnormal_min":np.nan,
               "auth_reported_abnormal_max":1.49,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

df5 = pd.read_excel(DATA_DIR/"Variant_score_files/MSH2_Jia_2021.xlsx", header = 0)
dc_5 = {"x": DATA_DIR/"Variant_score_files/MSH2_Jia_2021.xlsx", 
               "y": 0,
               "Dataset" :'MSH2_Jia_2021', 
               "Gene" : "MSH2", 
               "HGNC_id" : 7325, 
               "Chrom" : 2, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000251.2", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df5["Position"],
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df5['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df5['LOF score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df6 = pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", header = 0)
dc_6 = {"x": DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_WAF1nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df6['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df6['WAF1nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df7 = pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", header = 0)
dc_7 = {"x": DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_MDM2nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df7['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df7['MDM2nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}
df8 = pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", header = 0)
dc_8 = {"x": DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_BAXnWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df8['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df8['BAXnWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}
df9= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", header = 0)
dc_9 = {"x": DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_h1433snWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df9['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df9['h1433snWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df10= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", header = 0)
dc_10 = {"x": DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_AIP1nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df10['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df10['AIP1nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df11= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", header = 0)
dc_11 = {"x": DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_GADD45nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df11['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df11['GADD45nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df12= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", header = 0)
dc_12 = {"x": DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_NOXAnWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df12['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df12['NOXAnWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df13= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", header = 0)
dc_13 = {"x": DATA_DIR/"Variant_score_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_P53R2nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df13['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df13['P53R2nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df14= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Fortuno_2021.xlsx", header = 0)
dc_14 = {"x": DATA_DIR/"Variant_score_files/TP53_Fortuno_2021.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Fortuno_2021', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : 'NM_000546.5', 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df14['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df14['Fortuno_median'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df15= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_15 = {"x": DATA_DIR/"Variant_score_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_p53WT_Nutlin3', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df15["Position"],
               "aa_ref": df15["AA_wt"],
               "aa_alt": df15["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df15['A549_p53WT_Nutlin-3_Z-score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df16= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_16 = {"x": DATA_DIR/"Variant_score_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_p53null_Nutlin3', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df16["Position"],
               "aa_ref": df16["AA_wt"],
               "aa_alt": df16["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df16['A549_p53NULL_Nutlin-3_Z-score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df17= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_17 = {"x": DATA_DIR/"Variant_score_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_p53null_etoposide', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df17["Position"],
               "aa_ref": df17["AA_wt"],
               "aa_alt": df17["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df17['A549_p53NULL_Etoposide_Z-score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df18= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_18 = {"x": DATA_DIR/"Variant_score_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_combined_score', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df18["Position"],
               "aa_ref": df18["AA_wt"],
               "aa_alt": df18["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df18['Combined_score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#"Z" amino acids changed to "*", in Variant column where terminating amino acids are changed, Z is changed to "Ter", "B" is changed to "="
df19= pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Fayer_2021.xlsx", header = 1)
dc_19 = {"x": DATA_DIR/"Variant_score_files/TP53_Fayer_2021.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Fayer_2021_meta', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000546.5", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df19["Variant"],
               "consequence": np.nan,
               "auth_reported_score": df19['Classifier_prob_func_abnormal'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df19['Classifier_prediction'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df20= pd.read_excel(DATA_DIR/"Variant_score_files/PTEN_Mighell_2018.xlsx", header = 1)
dc_20 = {"x": DATA_DIR/"Variant_score_files/PTEN_Mighell_2018.xlsx", 
               "y": 1,
               "Dataset" :'PTEN_Mighell_2018', 
               "Gene" : "PTEN", 
               "HGNC_id" : 9588, 
               "Chrom" : 10, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000314.6", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df20["Variant (one letter)"],
               "consequence": np.nan,
               "auth_reported_score": df20['Cum_score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df20['High_conf'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df21= pd.read_excel(DATA_DIR/"Variant_score_files/PTEN_Matreyek_2018.xlsx", header = 0)
dc_21 = {"x": DATA_DIR/"Variant_score_files/PTEN_Matreyek_2018.xlsx", 
               "y": 0,
               "Dataset" :'PTEN_Matreyek_2018', 
               "Gene" : "PTEN", 
               "HGNC_id" : 9588, 
               "Chrom" : 10, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df21["hgvs_pro"],
               "consequence": np.nan,
               "auth_reported_score": df21['score'],
               "auth_reported_rep_score": df21[['score1','score2','score3','score4','score5',
                                                'score6','score7','score8']].fillna('').astype(str).agg(';'.join, axis=1).to_numpy(),
               "auth_reported_func_class": df21['functional_consequence'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#changed "X" to "*", nonsense
df22= pd.read_excel(DATA_DIR/"Variant_score_files/SCN5A_Glazer_2020.xlsx", header = 0)
dc_22 = {"x": DATA_DIR/"Variant_score_files/SCN5A_Glazer_2020.xlsx", 
               "y": 0,
               "Dataset" :'SCN5A_Glazer_2020', 
               "Gene" : "SCN5A", 
               "HGNC_id" : 10593, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "ENST00000333535", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df22["aa_num"],
               "aa_ref": df22["wt_allele"],
               "aa_alt": df22["mut_allele"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df22['dms'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df22["class"],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

df23= pd.read_excel(DATA_DIR/"Variant_score_files/VHL_Buckley_2024.xlsx", header = 2)
dc_23 = {"x": DATA_DIR/"Variant_score_files/VHL_Buckley_2024.xlsx", 
               "y": 2,
               "Dataset" :'VHL_Buckley_2024', 
               "Gene" : "VHL", 
               "HGNC_id" : 12687, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df23["hg38_pos"], 
               "ref_allele": df23["ref"], 
               "alt_allele": df23["alt"], 
               "auth_transcript_id" : "ENST00000256474.3", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df23["protPos"],
               "aa_ref": df23["oAA"],
               "aa_alt": df23["nAA"],
               "hgvs_c": df23["cHGVS"],
               "hgvs_p" : df23["pHGVS"],
               "consequence": np.nan,
               "auth_reported_score": df23['function_score_final'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df23["function_class"],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'Yes'}

#mutAA column "X" replaced with "*"
df24= pd.read_excel(DATA_DIR/"Variant_score_files/KCNH2_Kozek_Glazer_2020.xlsx", header = 1)
dc_24 = {"x": DATA_DIR/"Variant_score_files/KCNH2_Kozek_Glazer_2020.xlsx", 
               "y": 1,
               "Dataset" :'KCNH2_Kozek_Glazer_2020', 
               "Gene" : "KCNH2", 
               "HGNC_id" : 6251, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "ENST00000262186", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df24["resnum"],
               "aa_ref": df24["nativeAA"],
               "aa_alt": df24["mutAA"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df24['score.ave'],
               "auth_reported_rep_score": df24[['score.1', 'score.1']].fillna('').astype(str).agg(';'.join, axis=1).to_numpy(),
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#had to reformat excel file to run through script; reformatting done in excel
df25= pd.read_excel(DATA_DIR/"Variant_score_files/KCNH2_Jiang_2022.xlsx", header = 0)
dc_25 = {"x": DATA_DIR/"Variant_score_files/KCNH2_Jiang_2022.xlsx", 
               "y": 0,
               "Dataset" :'KCNH2_Jiang_2022', 
               "Gene" : "KCNH2", 
               "HGNC_id" : 6251, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000238.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df25["Protein Change"],
               "consequence": np.nan,
               "auth_reported_score": df25[' Normalised Current '],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#added 'p.' in the Human variant column
df26= pd.read_excel(DATA_DIR/"Variant_score_files/OTC_Lo_2023.xlsx", header = 1)
dc_26 = {"x": DATA_DIR/"Variant_score_files/OTC_Lo_2023.xlsx", 
               "y": 1,
               "Dataset" :'OTC_Lo_2023', 
               "Gene" : "OTC", 
               "HGNC_id" : 8512, 
               "Chrom" : "X", 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000531.6", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df26["Human_Variant"],
               "consequence": np.nan,
               "auth_reported_score": df26['Growth Estimate'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df26['Functional_Class'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#JAG1 supplement, added columns for aa_alt and aa_ref and functional class (according to author encoded functional class)
df27= pd.read_excel(DATA_DIR/"Variant_score_files/JAG1_Gilbert_2024.xlsx", header = 1)
dc_27 = {"x": DATA_DIR/"Variant_score_files/JAG1_Gilbert_2024.xlsx", 
               "y": 1,
               "Dataset" :'JAG1_Gilbert_2024', 
               "Gene" : "JAG1", 
               "HGNC_id" : 6188, 
               "Chrom" : 20, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df27["pos"], 
               "ref_allele": df27["Ref_Allele"], 
               "alt_allele": df27["Alt_Allele"], 
               "auth_transcript_id" : "NM_000214.3", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df27["AA_Position"],
               "aa_ref": df27["aa_ref"],
               "aa_alt": df27["aa_alt"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": df27["Consequence"],
               "auth_reported_score": df27['meanAcrossReps'],
               "auth_reported_rep_score": df27[['VarScore_1', 'VarScore_2','VarScore_3','VarScore_4','VarScore_5',
                                               'VarScore_6','VarScore_7']].fillna('').astype(str).agg(';'.join, axis=1).to_numpy(),
               "auth_reported_func_class": df27['func_class'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#BRCA2_Sahu_2023 supplement, new column added for hgvs_c without transcript annotation, "Intronic" removed from hgvs.p column
df28= pd.read_excel(DATA_DIR/"Variant_score_files/BRCA2_Sahu_2023_exon13.xlsx", header = 0)
dc_28 = {"x": DATA_DIR/"Variant_score_files/BRCA2_Sahu_2023_exon13.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_Sahu_2023_exon13_SGE', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df28["hgvs_c"],
               "hgvs_p" : df28["p.Nomenclature"],
               "consequence": np.nan,
               "auth_reported_score": df28['Function score DMSO'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": "nucleotide",
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

#BRCA2_Sahu_2023 supplement, new column added for hgvs_c without transcript annotation,"Intronic" removed from hgvs.p column
df29= pd.read_excel(DATA_DIR/"Variant_score_files/BRCA2_Sahu_2023_exon13.xlsx", header = 0)
dc_29 = {"x": DATA_DIR/"Variant_score_files/BRCA2_Sahu_2023_exon13.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_Sahu_2023_exon13_Cisplatin_Resistance', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df29["hgvs_c"],
               "hgvs_p" : df29["p.Nomenclature"],
               "consequence": np.nan,
               "auth_reported_score": df29['Function score cisplatin'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

#BRCA2_Sahu_2023 supplement, new column added for hgvs_c without transcript annotation,"Intronic" removed from hgvs.p column
df30= pd.read_excel(DATA_DIR/"Variant_score_files/BRCA2_Sahu_2023_exon13.xlsx", header = 0)
dc_30 = {"x": DATA_DIR/"Variant_score_files/BRCA2_Sahu_2023_exon13.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_Sahu_2023_exon13_Olaparib_Resistance', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df30["hgvs_c"],
               "hgvs_p" : df30["p.Nomenclature"],
               "consequence": np.nan,
               "auth_reported_score": df30['Function score olaparib'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

#assertions according to authors added as a new column in the excel sheet 
df31= pd.read_excel(DATA_DIR/"Variant_score_files/SCN5A_Ma_2024.xlsx", header = 0)
dc_31 = {"x": DATA_DIR/"Variant_score_files/SCN5A_Ma_2024.xlsx", 
               "y": 0,
               "Dataset" :'SCN5A_Ma_2024', 
               "Gene" : "SCN5A", 
               "HGNC_id" : 10593, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000335.5", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df31["Cell line"],
               "consequence": np.nan,
               "auth_reported_score": df31['CD sqrtNORM(-120mV) Mean'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df31['class'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}


df33 = pd.read_excel(DATA_DIR/"Variant_score_files/TSC2_IGVF.xlsx", header = 0)
dc_33 = {"x": DATA_DIR/"Variant_score_files/TSC2_IGVF.xlsx", 
               "y": 0,
               "Dataset" :'TSC2_IGVF', 
               "Gene" : "TSC2", 
               "HGNC_id" : 12363, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df33['aaChanges'],
               "consequence": np.nan,
               "auth_reported_score": df33['average'],
               "auth_reported_rep_score": df33[['abrep1','abrep2', 'abrep3']].fillna('').astype(str).agg(';'.join, axis=1).to_numpy(),
               "auth_reported_func_class": df33['functional_consequence'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#G6PD unppublished (Fowler lab)
df34 = pd.read_excel(DATA_DIR/"Variant_score_files/G6PD_IGVF.xlsx", header = 0)
dc_34 = {"x": DATA_DIR/"Variant_score_files/G6PD_IGVF.xlsx", 
               "y": 0,
               "Dataset" :'G6PD_IGVF', 
               "Gene" : "G6PD", 
               "HGNC_id" : 4057, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df34["variant_p"],
               "consequence": np.nan,
               "auth_reported_score": df34['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df34['functional_consequence'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#BARD1_IGVF (BBI)
df35 = pd.read_excel(DATA_DIR/"Variant_score_files/BARD1_IGVF.xlsx", header = 0)
dc_35 = {"x": DATA_DIR/"Variant_score_files/BARD1_IGVF.xlsx", 
               "y": 0,
               "Dataset" :'BARD1_IGVF', 
               "Gene" : "BARD1", 
               "HGNC_id" : 952, 
               "Chrom" : 2, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df35['pos'], 
               "ref_allele": df35['ref'], 
               "alt_allele": df35['alt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df35['amino_acid_change'],
               "consequence": df35['consequence'],
               "auth_reported_score": df35['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df35['functional_consequence'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}
#PALB2_IGVF (BBI)
df36 = pd.read_excel(DATA_DIR/"Variant_score_files/PALB2_IGVF.xlsx", header = 0)
dc_36 = {"x": DATA_DIR/"Variant_score_files/PALB2_IGVF.xlsx", 
               "y": 0,
               "Dataset" :'PALB2_IGVF', 
               "Gene" : "PALB2", 
               "HGNC_id" : 26144, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df36['pos'], 
               "ref_allele": df36['ref'], 
               "alt_allele": df36['alt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df36['amino_acid_change'],
               "consequence": df36['consequence'],
               "auth_reported_score": df36['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df36['functional_consequence'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#XRCC2_IGVF (BBI)
df37 = pd.read_excel(DATA_DIR/"Variant_score_files/XRCC2_IGVF.xlsx", header = 0)
dc_37 = {"x": DATA_DIR/"Variant_score_files/XRCC2_IGVF.xlsx", 
               "y": 0,
               "Dataset" :'XRCC2_IGVF', 
               "Gene" : "XRCC2", 
               "HGNC_id" : 12829, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df37['pos'], 
               "ref_allele": df37['ref'], 
               "alt_allele": df37['alt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df37['amino_acid_change'],
               "consequence": df37['consequence'],
               "auth_reported_score": df37['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df37['functional_consequence'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#SFPQ_IGVF (BBI)
df38 = pd.read_excel(DATA_DIR/"Variant_score_files/SFPQ_IGVF.xlsx", header = 0)
dc_38 = {"x": DATA_DIR/"Variant_score_files/SFPQ_IGVF.xlsx", 
               "y": 0,
               "Dataset" :'SFPQ_IGVF', 
               "Gene" : "SFPQ", 
               "HGNC_id" : 10774, 
               "Chrom" : 1, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df38['pos'], 
               "ref_allele": df38['ref'], 
               "alt_allele": df38['alt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df38['amino_acid_change'],
               "consequence": df38['consequence'],
               "auth_reported_score": df38['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df38['functional_consequence'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#BRCA2_IGVF (BBI)
df39 = pd.read_excel(DATA_DIR/"Variant_score_files/BRCA2_IGVF.xlsx", header = 0)
dc_39 = {"x": DATA_DIR/"Variant_score_files/BRCA2_IGVF.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_IGVF', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df39['pos'], 
               "ref_allele": df39['ref'], 
               "alt_allele": df39['alt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df39['amino_acid_change'],
               "consequence": df39['consequence'],
               "auth_reported_score": df39['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df39['functional_consequence'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}
#RAD51D_IGVF (BBI)
df40 = pd.read_excel(DATA_DIR/"Variant_score_files/RAD51D_IGVF.xlsx", header = 0)
dc_40 = {"x": DATA_DIR/"Variant_score_files/RAD51D_IGVF.xlsx", 
               "y": 0,
               "Dataset" :'RAD51D_IGVF', 
               "Gene" : "RAD51D", 
               "HGNC_id" : 9823, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df40['pos'], 
               "ref_allele": df40['ref'], 
               "alt_allele": df40['alt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df40['amino_acid_change'],
               "consequence": df40['consequence'],
               "auth_reported_score": df40['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df40['functional_consequence'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}
#CTCF_IGVF (BBI)
df41 = pd.read_excel(DATA_DIR/"Variant_score_files/CTCF_IGVF.xlsx", header = 0)
dc_41 = {"x": DATA_DIR/"Variant_score_files/CTCF_IGVF.xlsx", 
               "y": 0,
               "Dataset" :'CTCF_IGVF', 
               "Gene" : "CTCF", 
               "HGNC_id" : 13723, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df41['pos'], 
               "ref_allele": df41['ref'], 
               "alt_allele": df41['alt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df41['amino_acid_change'],
               "consequence": df41['consequence'],
               "auth_reported_score": df41['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df41['functional_consequence'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df42 = pd.read_excel(DATA_DIR/"Variant_score_files/BAP1_Waters_2024.xlsx", header = 0)
dc_42 = {"x": DATA_DIR/"Variant_score_files/BAP1_Waters_2024.xlsx", 
               "y": 2,
               "Dataset" :'BAP1_Waters_2024', 
               "Gene" : "BAP1", 
               "HGNC_id" : 950, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df42['pos'], 
               "ref_allele": df42['ref'], 
               "alt_allele": df42['alt'], 
               "auth_transcript_id" : 'ENST00000460680.6', 
               "transcript_pos" : df42['CDS_position'],
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df42['protein_position'],
               "aa_ref": df42['ref_aa'],
               "aa_alt": df42['alt_aa'],
               "hgvs_c": df42['HGVSc'],
               "hgvs_p" : np.nan,
               "consequence": df42['vep_consequence'],
               "auth_reported_score": df42['functional_score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df42['functional_classification'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#CALM1/2/3 yeast complementation assay, HGNC ID and other identifiers for CALM1 used here
df43 = pd.read_excel(DATA_DIR/"Variant_score_files/CALM1_CALM2_CALM3_Weile_2017.xlsx", header = 0)
dc_43 = {"x": DATA_DIR/"Variant_score_files/CALM1_CALM2_CALM3_Weile_2017.xlsx", 
               "y": 0,
               "Dataset" :'CALM1_CALM2_CALM3_Weile_2017', 
               "Gene" : "CALM1_CALM2_CALM3", 
               "HGNC_id" : 1442, 
               "Chrom" : 14, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" :np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df43['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df43['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#TPK1 yeast complementation assay 
df44 = pd.read_excel(DATA_DIR/"Variant_score_files/TPK1_Weile_2017.xlsx", header = 0)
dc_44 = {"x": DATA_DIR/"Variant_score_files/TPK1_Weile_2017.xlsx", 
               "y": 0,
               "Dataset" :'TPK1_Weile_2017', 
               "Gene" : "TPK1", 
               "HGNC_id" : 17358, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" :np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df44['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df44['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#DDX3X SGE
df45 = pd.read_excel(DATA_DIR/"Variant_score_files/DDX3X_Radford_2023.xlsx", header = 0)
dc_45 = {"x": DATA_DIR/"Variant_score_files/DDX3X_Radford_2023.xlsx", 
               "y": 0,
               "Dataset" :'DDX3X_Radford_2023', 
               "Gene" : "DDX3X", 
               "HGNC_id" : 16393, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : df45['VCF_position'], 
               "ref_allele": df45['VCF_Ref'], 
               "alt_allele": df45['VCF_Alt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : df45['CDS_position'],
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df45['Protein_position'],
               "aa_ref": df45['ref_aa'],
               "aa_alt": df45['alt_aa'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": df45['Consequence'],
               "auth_reported_score": df45['D15_combined_LFC'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df45['SGE_functional_classification'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}


#HMBS ubiquitous isoform

df46 = pd.read_excel(DATA_DIR/"Variant_score_files/HMBS_van_Loggerenberg_2023_ubquitous.xlsx", header = 0)
dc_46 = {"x": DATA_DIR/"Variant_score_files/HMBS_van_Loggerenberg_2023_ubquitous.xlsx", 
               "y": 0,
               "Dataset" :'HMBS_van_Loggerenberg_2023_ubquitous', 
               "Gene" : "HMBS", 
               "HGNC_id" : 4982, 
               "Chrom" : 11, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df46['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df46['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#HMBS erythroid

df47 = pd.read_excel(DATA_DIR/"Variant_score_files/HMBS_van_Loggerenberg_2023_erythroid.xlsx", header = 0)
dc_47 = {"x": DATA_DIR/"Variant_score_files/HMBS_van_Loggerenberg_2023_erythroid.xlsx", 
               "y": 0,
               "Dataset" :'HMBS_van_Loggerenberg_2023_erythroid', 
               "Gene" : "HMBS", 
               "HGNC_id" : 4982, 
               "Chrom" : 11, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df47['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df47['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df48 = pd.read_excel(DATA_DIR/"Variant_score_files/HMBS_van_Loggerenberg_2023_combined.xlsx", header = 0)
dc_48 = {"x": DATA_DIR/"Variant_score_files/HMBS_van_Loggerenberg_2023_combined.xlsx", 
               "y": 0,
               "Dataset" :'HMBS_van_Loggerenberg_2023_combined', 
               "Gene" : "HMBS", 
               "HGNC_id" : 4982, 
               "Chrom" : 11, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df48['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df48['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#added modified hgvs_c column to dataset, took out extra information from bottom, turned 'synonymous' variants in protein description column
#into their respective p.
df49 = pd.read_excel(DATA_DIR/"Variant_score_files/RHO_Wan_2019.xlsx", header = 0)
dc_49 = {"x": DATA_DIR/"Variant_score_files/RHO_Wan_2019.xlsx", 
               "y": 0,
               "Dataset" :'RHO_Wan_2019', 
               "Gene" : "RHO", 
               "HGNC_id" : 10012, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df49['hgvs_c'],
               "hgvs_p" : df49['Protein description (HGVS RHO_v001:p. or NP_000530.1:p.)'],
               "consequence": np.nan,
               "auth_reported_score": df49['NGS-based surface RHO surface assay, mean'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df49['RHO surface expression, final category'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}


#Carboxylation-sensitive FIX-specific antibody score, X replaced with * for terminating aa

df50 = pd.read_excel(DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", header = 0)
dc_50 = {"x": DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2025_carboxy_F9_specific', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df50['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df50['Carboxylation-sensitive FIX-specific antibody score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df50['Functional_consequence_F9_Popp_2025_carboxy_F9_specific'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df59 = pd.read_excel(DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", header = 0)
dc_59 = {"x": DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2025_model', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df59['Variant'],
               "consequence": np.nan,
               "auth_reported_score": np.nan,
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df59['Model prediction'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df51 = pd.read_excel(DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", header = 0)
dc_51 = {"x": DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2025_heavy_chain', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df51['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df51['Heavy chain antibody score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df51['Functional_consequence_F9_Popp_2025_heavy_chain'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df52 = pd.read_excel(DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", header = 0)
dc_52 = {"x": DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2025_light_chain', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df52['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df52['Light chain antibody score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df52['Functional_consequence_F9_Popp_2025_light_chain'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df53 = pd.read_excel(DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", header = 0)
dc_53 = {"x": DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2025_carboxy_gla_motif', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df53['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df53['Carboxylation-sensitive Gla-motif antibody score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df54 = pd.read_excel(DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", header = 0)
dc_54 = {"x": DATA_DIR/"Variant_score_files/F9_Popp_2025.xlsx", 
               "y": 0,
               "Dataset" :'F9_Popp_2025_strep_2', 
               "Gene" : "F9", 
               "HGNC_id" : 3551, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df54['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df54['Strep II tag antibody score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df54['Functional_consequence_F9_Popp_2025_strep_2'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}
#GCK

df55 = pd.read_excel(DATA_DIR/"Variant_score_files/GCK_Gersing_2023_complementation.xlsx", header = 15)
dc_55 = {"x": DATA_DIR/"Variant_score_files/GCK_Gersing_2023_complementation.xlsx", 
               "y": 15,
               "Dataset" :'GCK_Gersing_2023_complementation', 
               "Gene" : "GCK", 
               "HGNC_id" : 4195, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df55['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df55['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df57 = pd.read_excel(DATA_DIR/"Variant_score_files/ASPA_Grønbæk-Thygesen_2024.xlsx", header = 0)
dc_57 = {"x": DATA_DIR/"Variant_score_files/ASPA_Grønbæk-Thygesen_2024.xlsx", 
               "y": 0,
               "Dataset" :'ASPA_Grønbæk-Thygesen_2024_abundance', 
               "Gene" : "ASPA", 
               "HGNC_id" : 756, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df57['variant'],
               "consequence": np.nan,
               "auth_reported_score": df57['abundance_score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}


df58 = pd.read_excel(DATA_DIR/"Variant_score_files/ASPA_Grønbæk-Thygesen_2024.xlsx", header = 0)
dc_58 = {"x": DATA_DIR/"Variant_score_files/ASPA_Grønbæk-Thygesen_2024.xlsx", 
               "y": 0,
               "Dataset" :'ASPA_Grønbæk-Thygesen_2024_toxicity', 
               "Gene" : "ASPA", 
               "HGNC_id" : 756, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df58['variant'],
               "consequence": np.nan,
               "auth_reported_score": df58['tox_score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df61 = pd.read_excel(DATA_DIR/"Variant_score_files/RAD51C_Olvera-León_2024.xlsx", header = 0)
dc_61 = {"x": DATA_DIR/"Variant_score_files/RAD51C_Olvera-León_2024.xlsx", 
               "y": 2,
               "Dataset" :'RAD51C_Olvera-León_2024', 
               "Gene" : "RAD51C", 
               "HGNC_id" : 9820, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df61['pos'], 
               "ref_allele": df61['ref'], 
               "alt_allele": df61['alt'], 
               "auth_transcript_id" : "ENST00000337432.9", 
               "transcript_pos" : df61['CDS_position'],
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df61['protein_position'],
               "aa_ref": df61['ref_aa'],
               "aa_alt": df61['alt_aa'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": df61['vep_consequence'],
               "auth_reported_score": df61['z_score_D4_D14'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df61['functional_classification'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

# changed 'Intronic' in AA.change to blanks
df63 = pd.read_excel(DATA_DIR/"Variant_score_files/BRCA2_Sahu_2025_SGE.xlsx", header = 1)
dc_63 = {"x": DATA_DIR/"Variant_score_files/BRCA2_Sahu_2025_SGE.xlsx", 
               "y": 1,
               "Dataset" :'BRCA2_Sahu_2025_SGE', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df63['g.nom'].str.extract(r'g\.(\d+)').astype(int), 
               "ref_allele": df63['Nucleotide.change'].str.extract(r'^([ACGT])[:>]'), 
               "alt_allele": df63['Nucleotide.change'].str.extract(r'[:>]([ACGT])$'), 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : df63['c.nom'].str.extract(r'c\.([\d\+\-]+)(?=[A-Za-z])'),
               "transcript_ref" : df63['c.nom'].str.extract(r'\d+([A-Z])'), 
               "transcript_alt" : df63['c.nom'].str.extract(r'>([A-Z])'),
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df63['c.nom'].str.extract(r'(c\..*)'),
               "hgvs_p" : df63['AA.change'],
               "consequence": np.nan,
               "auth_reported_score": df63['Function.score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df64 = pd.read_excel(DATA_DIR/"Variant_score_files/GCK_Gersing_2024_abundance.xlsx", header = 0)
dc_64 = {"x": DATA_DIR/"Variant_score_files/GCK_Gersing_2024_abundance.xlsx", 
               "y": 0,
               "Dataset" :'GCK_Gersing_2024_abundance', 
               "Gene" : "GCK", 
               "HGNC_id" : 4195, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "ENST00000403799.8", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df64['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df64['score_abundance'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df65 = pd.read_excel(DATA_DIR/"Variant_score_files/KCNH2_O_Neill_2024_surface_expression.xlsx", header = 2)
dc_65 = {"x": DATA_DIR/"Variant_score_files/KCNH2_O_Neill_2024_surface_expression.xlsx", 
               "y": 2,
               "Dataset" :"KCNH2_O_Neill_2024_surface_expression", 
               "Gene" : "KCNH2", 
               "HGNC_id" : 6251, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "ENST00000262186", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df65['HGSV'],
               "consequence": np.nan,
               "auth_reported_score": df65['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df66 = pd.read_excel(DATA_DIR/"Variant_score_files/PAX6_McDonnell_2024_LE9_geneticin.xlsx", header = 0)
dc_66 = {"x": DATA_DIR/"Variant_score_files/PAX6_McDonnell_2024_LE9_geneticin.xlsx", 
               "y": 0,
               "Dataset" :"PAX6_McDonnell_2024_LE9_geneticin", 
               "Gene" : "PAX6", 
               "HGNC_id" : 8620, 
               "Chrom" : 11, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df66['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df66['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df67 = pd.read_excel(DATA_DIR/"Variant_score_files/PAX6_McDonnell_2024_LE9_no_geneticin.xlsx", header = 0)
dc_67 = {"x": DATA_DIR/"Variant_score_files/PAX6_McDonnell_2024_LE9_no_geneticin.xlsx", 
               "y": 0,
               "Dataset" :"PAX6_McDonnell_2024_LE9_no_geneticin", 
               "Gene" : "PAX6", 
               "HGNC_id" : 8620, 
               "Chrom" : 11, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df67['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df67['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df68 = pd.read_excel(DATA_DIR/"Variant_score_files/PAX6_McDonnell_2024_BLX_geneticin.xlsx", header = 0)
dc_68 = {"x": DATA_DIR/"Variant_score_files/PAX6_McDonnell_2024_BLX_geneticin.xlsx", 
               "y": 0,
               "Dataset" :"PAX6_McDonnell_2024_BLX_geneticin", 
               "Gene" : "PAX6", 
               "HGNC_id" : 8620, 
               "Chrom" : 11, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df68['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df68['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df69 = pd.read_excel(DATA_DIR/"Variant_score_files/PAX6_McDonnell_2024_BLX_no_geneticin.xlsx", header = 0)
dc_69 = {"x": DATA_DIR/"Variant_score_files/PAX6_McDonnell_2024_BLX_no_geneticin.xlsx", 
               "y": 0,
               "Dataset" :"PAX6_McDonnell_2024_BLX_no_geneticin", 
               "Gene" : "PAX6", 
               "HGNC_id" : 8620, 
               "Chrom" : 11, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df69['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df69['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#BRCA2_Sahu_2023 supplement, new column added for hgvs_c without transcript annotation, "Intronic" removed from hgvs.p column
df70= pd.read_excel(DATA_DIR/"Variant_score_files/BRCA2_Sahu_2023_exon13.xlsx", header = 0)
dc_70 = {"x": DATA_DIR/"Variant_score_files/BRCA2_Sahu_2023_exon13.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_Sahu_2023_exon13_global_score', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df70["hgvs_c"],
               "hgvs_p" : df70["p.Nomenclature"],
               "consequence": np.nan,
               "auth_reported_score": df28['Global function score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": "nucleotide",
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

df71 = pd.read_excel(DATA_DIR/"Variant_score_files/CBS_Sun_2020_high_B6.xlsx", header = 1)
dc_71 = {"x": DATA_DIR/"Variant_score_files/CBS_Sun_2020_high_B6.xlsx", 
               "y": 1,
               "Dataset" :'CBS_Sun_2020_high_B6', 
               "Gene" : "CBS", 
               "HGNC_id" : 1550, 
               "Chrom" : 21, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df71['nucleotide'],
               "hgvs_p" : df71['amino acid'],
               "consequence": np.nan,
               "auth_reported_score": df71['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df72 = pd.read_excel(DATA_DIR/"Variant_score_files/CBS_Sun_2020_low_B6.xlsx", header = 1)
dc_72 = {"x": DATA_DIR/"Variant_score_files/CBS_Sun_2020_low_B6.xlsx", 
               "y": 1,
               "Dataset" :'CBS_Sun_2020_low_B6', 
               "Gene" : "CBS", 
               "HGNC_id" : 1550, 
               "Chrom" : 21, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df72['nucleotide'],
               "hgvs_p" : df72['amino acid'],
               "consequence": np.nan,
               "auth_reported_score": df72['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df73 = pd.read_excel(DATA_DIR/"Variant_score_files/KCNQ4_Zheng_2022_current_homozygous.xlsx", header = 2)
dc_73 = {"x": DATA_DIR/"Variant_score_files/KCNQ4_Zheng_2022_current_homozygous.xlsx", 
               "y": 2,
               "Dataset" :'KCNQ4_Zheng_2022_current_homozygous', 
               "Gene" : "KCNQ4", 
               "HGNC_id" : 6298, 
               "Chrom" : 1, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df73['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df73['Normalized current (Imut/Iwt, at +40 mV)'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df74 = pd.read_excel(DATA_DIR/"Variant_score_files/KCNQ4_Zheng_2022_v12_homozygous.xlsx", header = 2)
dc_74 = {"x": DATA_DIR/"Variant_score_files/KCNQ4_Zheng_2022_v12_homozygous.xlsx", 
               "y": 2,
               "Dataset" :'KCNQ4_Zheng_2022_v12_homozygous', 
               "Gene" : "KCNQ4", 
               "HGNC_id" : 6298, 
               "Chrom" : 1, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df74['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df74['Δ V1/2 (mV)'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df75 = pd.read_excel(DATA_DIR/"Variant_score_files/CRX_Shepherdson_2024.xlsx", header = 1)
dc_75 = {"x": DATA_DIR/"Variant_score_files/CRX_Shepherdson_2024.xlsx", 
               "y": 1,
               "Dataset" :'CRX_Shepherdson_2024', 
               "Gene" : "CRX", 
               "HGNC_id" : 2383, 
               "Chrom" : 19, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df75['var_pos'],
               "aa_ref": df75['var_ref'],
               "aa_alt": df75['var_alt'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df75['normalized_activity_score_mean'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df75['classification'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

df76 = pd.read_excel(DATA_DIR/"Variant_score_files/CHK2_Gebbia_2024.xlsx", header = 0)
dc_76 = {"x": DATA_DIR/"Variant_score_files/CHK2_Gebbia_2024.xlsx", 
               "y": 0,
               "Dataset" :'CHEK2_Gebbia_2024', 
               "Gene" : "CHEK2", 
               "HGNC_id" : 16627, 
               "Chrom" : 22, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df76['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df76['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#changed 'X' amino acid to '*' for nonsense
df77 = pd.read_excel(DATA_DIR/"Variant_score_files/KCNE1_Muhammad_2024.xlsx", header = 0)
dc_77 = {"x": DATA_DIR/"Variant_score_files/KCNE1_Muhammad_2024.xlsx", 
               "y": 0,
               "Dataset" :'KCNE1_Muhammad_2024_potassium_flux', 
               "Gene" : "KCNE1", 
               "HGNC_id" : 6240, 
               "Chrom" : 21, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : 'NM_001127670.4', 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df77['aapos'],
               "aa_ref": df77['aaref'],
               "aa_alt": df77['aaalt'],
               "hgvs_c": np.nan,
               "hgvs_p" : df77['mutation'],
               "consequence": np.nan,
               "auth_reported_score": df77['FuncScore (Mean)'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df77['Functional Score Category'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'X' amino acid to '*' for nonsense
df78 = pd.read_excel(DATA_DIR/"Variant_score_files/KCNE1_Muhammad_2024.xlsx", header = 0)
dc_78 = {"x": DATA_DIR/"Variant_score_files/KCNE1_Muhammad_2024.xlsx", 
               "y": 0,
               "Dataset" :'KCNE1_Muhammad_2024_trafficking_WT_background_DN', 
               "Gene" : "KCNE1", 
               "HGNC_id" : 6240, 
               "Chrom" : 21, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : 'NM_001127670.4', 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df78['aapos'],
               "aa_ref": df78['aaref'],
               "aa_alt": df78['aaalt'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df78['TrafScore (Mean)'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df78['Trafficking Score Category'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'X' amino acid to '*' for nonsense
df79 = pd.read_excel(DATA_DIR/"Variant_score_files/KCNE1_Muhammad_2024.xlsx", header = 0)
dc_79 = {"x": DATA_DIR/"Variant_score_files/KCNE1_Muhammad_2024.xlsx", 
               "y": 0,
               "Dataset" :'KCNE1_Muhammad_2024_trafficking', 
               "Gene" : "KCNE1", 
               "HGNC_id" : 6240, 
               "Chrom" : 21, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : 'NM_001127670.4', 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df79['aapos'],
               "aa_ref": df79['aaref'],
               "aa_alt": df79['aaalt'],
               "hgvs_c": np.nan,
               "hgvs_p" : df79['mutation'],
               "consequence": np.nan,
               "auth_reported_score": df79['TrafScore_E1only (Mean)'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df79['Trafficking Score Category (E1 only)'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

df80 = pd.read_excel(DATA_DIR/"Variant_score_files/SGCB_Li_2023.xlsx", header = 0)
dc_80 = {"x": DATA_DIR/"Variant_score_files/SGCB_Li_2023.xlsx", 
               "y": 0,
               "Dataset" :'SGCB_Li_2023', 
               "Gene" : "SGCB", 
               "HGNC_id" : 10806, 
               "Chrom" : 4, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df80['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df80['Functional_Score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df80['Functional_Score_Prediction'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df81 = pd.read_excel(DATA_DIR/"Variant_score_files/NDUFAF6_Sung_2024.xlsx", header = 0)
dc_81 = {"x": DATA_DIR/"Variant_score_files/NDUFAF6_Sung_2024.xlsx", 
               "y": 0,
               "Dataset" :'NDUFAF6_Sung_2024', 
               "Gene" : "NDUFAF6", 
               "HGNC_id" : 28625, 
               "Chrom" : 8, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df81['Position'],
               "aa_ref": df81['Reference AA'],
               "aa_alt": df81['Variant AA'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df81['Fitness'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df81['Functional Classification'],
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed NA in start-loss to Del
df82 = pd.read_excel(DATA_DIR/"Variant_score_files/FKRP_Ma_2024.xlsx", header = 0)
dc_82 = {"x": DATA_DIR/"Variant_score_files/FKRP_Ma_2024.xlsx", 
               "y": 0,
               "Dataset" :'FKRP_Ma_2024', 
               "Gene" : "FKRP", 
               "HGNC_id" : 17997, 
               "Chrom" : 19, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df82['hg38_site'], 
               "ref_allele": df82['WT_nt'], 
               "alt_allele": df82['Variant_nt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df82['codon'],
               "aa_ref": df82['WT_AA'],
               "aa_alt": df82['Variant_AA'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": df82['classification'],
               "auth_reported_score": df82['fitness_normalized'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df82['SMuRF_reclass'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}


#changed NA in start-loss to Del, genomic ref and alts are incorrect, they have been corrected and updated
df83 = pd.read_excel(DATA_DIR/"Variant_score_files/LARGE1_Ma_2024.xlsx", header = 0)
dc_83 = {"x": DATA_DIR/"Variant_score_files/LARGE1_Ma_2024.xlsx", 
               "y": 0,
               "Dataset" :'LARGE1_Ma_2024', 
               "Gene" : "LARGE1", 
               "HGNC_id" : 6511, 
               "Chrom" : 22, 
               "hg19_pos" : np.nan, 
               "hg38_start" : df83['hg38_site'], 
               "ref_allele": df83['corrected_ref'], 
               "alt_allele": df83['corrected_alt'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df83['codon'],
               "aa_ref": df83['WT_AA'],
               "aa_alt": df83['Variant_AA'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": df83['classification'],
               "auth_reported_score": df83['fitness_normalized'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df83['SMuRF_label'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#TP53_Boettcher changed Z to '*' and B to '=', calculated average scores for Nutlin_Ratio_average from both replicates
df84 = pd.read_excel(DATA_DIR/"Variant_score_files/TP53_Boettcher_2019.xlsx", header = 0)
dc_84 = {"x": DATA_DIR/"Variant_score_files/TP53_Boettcher_2019.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Boettcher_2019', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df84['POS'],
               "aa_ref": df84['Wt_aa'],
               "aa_alt": df84['Vt_aa'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df84['Nutlin_Ratio_average'],
               "auth_reported_rep_score": df84[['R1_Nutlin_Ratio_Lo_Hi','R3_Nutlin_Ratio_Lo_Hi']].fillna('').astype(str).agg(';'.join, axis=1).to_numpy(),
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

df85 = pd.read_excel(DATA_DIR/"Variant_score_files/CARD11_Meitlis_2020_SGE_Ibrutinib_GoF.xlsx", header = 0)
dc_85 = {"x": DATA_DIR/"Variant_score_files/CARD11_Meitlis_2020_SGE_Ibrutinib_GoF.xlsx", 
               "y": 0,
               "Dataset" :'CARD11_Meitlis_2020_SGE_Ibrutinib_GoF', 
               "Gene" : "CARD11", 
               "HGNC_id" : 16393, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df85['hgvs_nt'],
               "hgvs_p" : df85['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df85['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df85['functional_class'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

df86 = pd.read_excel(DATA_DIR/"Variant_score_files/CARD11_Meitlis_2020_SGE_LoF.xlsx", header = 0)
dc_86 = {"x": DATA_DIR/"Variant_score_files/CARD11_Meitlis_2020_SGE_LoF.xlsx", 
               "y": 0,
               "Dataset" :'CARD11_Meitlis_2020_SGE_LoF', 
               "Gene" : "CARD11", 
               "HGNC_id" : 16393, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df86['hgvs_nt'],
               "hgvs_p" : df86['hgvs_pro'],
               "consequence": np.nan,
               "auth_reported_score": df86['score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df86['functional_class'],
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

df87 = pd.read_excel(DATA_DIR/"Variant_score_files/TARDBP_Bolognesi_Faure_2019.xlsx", header = 0)
dc_87 = {"x": DATA_DIR/"Variant_score_files/TARDBP_Bolognesi_Faure_2019.xlsx", 
               "y": 0,
               "Dataset" :'TARDBP_Bolognesi_Faure_2019', 
               "Gene" : "TARDBP", 
               "HGNC_id" : 11571, 
               "Chrom" : 1, 
               "hg19_pos" : np.nan, 
               "hg38_start" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df87['Pos_abs'],
               "aa_ref": df87['WT_AA'],
               "aa_alt": df87['Mut'],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df87['toxicity'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}


dc_list = [dc_1, dc_2, dc_3,dc_4,dc_5,dc_6,dc_7,dc_8,dc_9,dc_10,dc_11,dc_12,dc_13,dc_14,dc_15,
           dc_16,dc_17,dc_18,dc_19,dc_20,dc_21,dc_22,dc_23, dc_24,dc_25,dc_26,dc_27,dc_28,dc_29,dc_30,
          dc_31,dc_33,dc_34,dc_35,dc_36,dc_37,dc_38,dc_39,dc_40,dc_41,dc_42,dc_43,dc_44,dc_45,dc_46,
          dc_47, dc_48,dc_49,dc_50,dc_51,dc_52,dc_53,dc_54,dc_55,dc_57,dc_58,dc_59,
          dc_61,dc_63, dc_64, dc_65,dc_66,dc_67,dc_68,dc_69,dc_70,dc_71,dc_72,dc_73,dc_74,dc_75,dc_76,
           dc_77,dc_78,dc_79,dc_80,dc_81,dc_82,dc_83,dc_84,dc_85,dc_86,dc_87]

final_combined_df = data_harmonization_loop(dc_list)

/net/gs/vol3/software/modules-sw/python/3.12.1/Linux/Ubuntu22.04/x86_64/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


No match for row PTEN_Matreyek_2018-3789 with value _wt


/net/gs/vol3/software/modules-sw/python/3.12.1/Linux/Ubuntu22.04/x86_64/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


No match for row SCN5A_Ma_2024-0 with value SCN5A
No match for row G6PD_IGVF-6709 with value p.=
No match for row BARD1_IGVF-0 with value ---
No match for row BARD1_IGVF-1 with value ---
No match for row BARD1_IGVF-2 with value ---
No match for row BARD1_IGVF-3 with value ---
No match for row BARD1_IGVF-4 with value ---
No match for row BARD1_IGVF-5 with value ---
No match for row BARD1_IGVF-6 with value ---
No match for row BARD1_IGVF-7 with value ---
No match for row BARD1_IGVF-8 with value ---
No match for row BARD1_IGVF-9 with value ---
No match for row BARD1_IGVF-10 with value ---
No match for row BARD1_IGVF-11 with value ---
No match for row BARD1_IGVF-12 with value ---
No match for row BARD1_IGVF-13 with value ---
No match for row BARD1_IGVF-14 with value ---
No match for row BARD1_IGVF-15 with value ---
No match for row BARD1_IGVF-16 with value ---
No match for row BARD1_IGVF-17 with value ---
No match for row BARD1_IGVF-18 with value ---
No match for row BARD1_IGVF-19 with val

In [166]:
#drop lines without a functional score 

final_combined_df = final_combined_df[
    ((final_combined_df['Dataset'] != 'F9_Popp_2025_model') & 
     (~final_combined_df['auth_reported_score'].isna())) |
    ((final_combined_df['Dataset'] == 'F9_Popp_2025_model') & 
     (~final_combined_df['auth_reported_func_class'].isna()))
]

# for OTC_Lo drop variants in the SMG loop, referred to authors as not useable 
final_combined_df = final_combined_df[final_combined_df['auth_reported_func_class'] != 'SMG Loop']

#give each variant a unique identifier
final_combined_df['ID'] = final_combined_df.apply(lambda row: f"{row['Dataset']}_var{row.name + 1}", axis=1)

# Save intermediate combined dataframe (temporary output)
tmp_file = TMP_DIR / "supp_data_combined.csv.gz"
final_combined_df.to_csv(tmp_file, compression = 'gzip', index=False)

print(f"Temporary file written to: {tmp_file}")


Temporary file written to: /net/bbi/vol1/home/mtejura/IGVF-cvfg-pillar-project/tmp/supp_data_combined.csv.gz


In [167]:
ID_length = len(final_combined_df['ID'].unique())

In [168]:
import pandas as pd
p_data = pd.read_csv(TMP_DIR / "supp_data_combined.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2598068743.py:2: DtypeWarning: Columns (3,7,8,9,10,11,12,13,16,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  p_data = pd.read_csv(TMP_DIR / "supp_data_combined.csv.gz")


In [169]:
#load big curation sheet for merge
curation_data = pd.read_excel(DATA_DIR / "Supplemental_Data/Supplementary_Data_3.xlsx", sheet_name="Curation", header=0)

#merge and add additional information from the big curation sheet
df_merge = pd.merge(p_data, curation_data[['Dataset Name', 'Ensembl Transcript ID', 
                                           'RefSeq Transcript ID','Interval 1 Name', 'Interval 1 Range',
                                           'Interval 1 Class', 'Interval 2 Name', 'Interval 2 Range',
                                           'Interval 2 Class', 'Interval 3 Name', 'Interval 3 Range',
                                           'Interval 3 Class', 'Interval 4 Name', 'Interval 4 Range',
                                           'Interval 4 Class', 'Interval 5 Name', 'Interval 5 Range',
                                           'Interval 5 Class', 'Interval 6 Name', 'Interval 6 Range','Interval 6 Class']], 
                                           left_on='Dataset', right_on = "Dataset Name", how='left')
df_merge = df_merge.drop_duplicates()

df_merge.to_csv(TMP_DIR / "supp_data_combined_curation.csv.gz", compression = 'gzip', index = False)

In [170]:
# --- Validation Block: Merge Integrity Check ---

# Check unique IDs
ids_before = set(p_data['ID'].dropna().unique())
ids_after = set(df_merge['ID'].dropna().unique())

if ids_before != ids_after:
    missing_ids = ids_before - ids_after
    new_ids = ids_after - ids_before
    raise ValueError(
        f"[ERROR] Merge altered unique IDs.\n"
        f"Missing IDs after merge: {missing_ids}\n"
        f"New unexpected IDs after merge: {new_ids}\n"
        f"Count before: {len(ids_before)}, after: {len(ids_after)}"
    )
else:
    print(f"[OK] Unique IDs preserved across merge (n={len(ids_before)}).")

# Check DataFrame lengths
len_before = len(p_data)
len_after = len(df_merge)

if len_before != len_after:
    raise ValueError(
        f"[ERROR] Merge changed DataFrame length.\n"
        f"Length before: {len_before}, Length after: {len_after}."
    )
else:
    print(f"[OK] DataFrame length preserved across merge (n={len_before}).")

[OK] Unique IDs preserved across merge (n=434166).
[OK] DataFrame length preserved across merge (n=434166).


In [171]:
import pandas as pd
import numpy as np
import re

# -----------------------------------------------------------------------------
# KEY GENERATION 
#
# Goal:
#   Create a standardized variant "key" for each row that can be used to join with VEP output from Ensembl VEP,
#
# Dataset types:
#   1) datasets_with_chromosome_key:
#        Use genomic coordinates (hg38) to build a RefSeq-style HGVS g. key:
#        "NC_0000{chrom}:g.{pos}{ref}>{alt}"
#        (Chromosomes X/Y are mapped to 23/24 for formatting consistency.)
#
#   2) cdot_datasets:
#        These datasets already provide HGVS c. notation, so we generate:
#        "{ENST}:{hgvs_c}"
#        where transcript versions are stripped (ENSTxxxxxx.Y -> ENSTxxxxxx)
#        to avoid mismatches caused purely by transcript versioning.
#
#   3) all other datasets:
#        These typically provide protein-level identifiers (aa_ref/aa_pos/aa_alt)
#        rather than explicit nucleotide alleles. We map nucleotide-level
#        keys by expanding the alternate amino acid into all possible codons
#        (amino_acid_to_codon), and then generating one c. key per codon:
#        "{ENST}:c.{nucpos}_{nucpos+2}delins{codon}"
#
#        IMPORTANT: This produces multiple possible keys per protein variant
#        because codon identity is not known from amino acid changes alone.
#
# Output:
#   A new column "key" added to the dataset. Some rows expand into multiple rows
#   (one per possible codon). Rows that fail key generation are assigned
#   "placeholder_key" and logged as warnings for follow-up QC.
# -----------------------------------------------------------------------------


# Load data
pd_1 = pd.read_csv(TMP_DIR / "supp_data_combined_curation.csv.gz", header=0)

# Amino acid to codon dictionary
amino_acid_to_codon = {
    'F': ['TTT', 'TTC'], 'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
    'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'], 'Y': ['TAT', 'TAC'],
    'Ter': ['TAA', 'TAG', 'TGA'], '*': ['TAA', 'TAG', 'TGA'], 'C': ['TGT', 'TGC'],
    'W': ['TGG'], 'P': ['CCT', 'CCC', 'CCA', 'CCG'], 'H': ['CAT', 'CAC'],
    'Q': ['CAA', 'CAG'], 'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'I': ['ATT', 'ATC', 'ATA'], 'M': ['ATG'], 'T': ['ACT', 'ACC', 'ACA', 'ACG'],
    'N': ['AAT', 'AAC'], 'K': ['AAA', 'AAG'], 'V': ['GTT', 'GTC', 'GTA', 'GTG'],
    'A': ['GCT', 'GCC', 'GCA', 'GCG'], 'D': ['GAT', 'GAC'], 'E': ['GAA', 'GAG'],
    'G': ['GGT', 'GGC', 'GGA', 'GGG'], '-': ['del'], 'Del': ['del'], 'X' : ['del']
}

datasets_with_chromosome_key = {
    "BARD1_IGVF", "CTCF_IGVF", 'BRCA2_IGVF',
    'PALB2_IGVF','RAD51D_IGVF','SFPQ_IGVF','XRCC2_IGVF',
    'BAP1_Waters_2024','DDX3X_Radford_2023',"BRCA2_Sahu_2025_SGE",
    "RAD51C_Olvera-León_2024",'NPC1_Erwood_2022_RPE1',
    "NPC1_Erwood_2022_HEK293T",'LARGE1_Ma_2024','FKRP_Ma_2024','BRCA2_Erwood_2022', 'JAG1_Gilbert_2024'
}

cdot_datasets = [
    'BRCA1_Findlay_2018','BRCA2_Hu_2024',"VHL_Buckley_2024","BRCA2_Sahu_2023_exon13_SGE","BRCA2_Sahu_2023_exon13_global_score",
    "BRCA2_Sahu_2023_exon13_Cisplatin_Resistance","BRCA2_Sahu_2023_exon13_Olaparib_Resistance","RHO_Wan_2019",
    "CBS_Sun_2020_high_B6","CBS_Sun_2020_low_B6",'CARD11_Meitlis_2020_SGE_LoF',
    'CARD11_Meitlis_2020_SGE_Ibrutinib_DN' 
]


def generate_key(row):
    dataset = row['Dataset']
    try:
        if dataset in datasets_with_chromosome_key:
            chrom = row['Chrom']
            pos = int(row['hg38_start'])
            ref = row['ref_allele']
            alt = row['alt_allele']
            chrom_str = str(chrom).upper()
            if chrom_str == "X":
                chrom_fmt = "23"
            elif chrom_str == "Y":
                chrom_fmt = "24"
            elif chrom_str in ["MT", "M"]:
                chrom_fmt = "MT" 
            else:
                chrom_int = int(chrom_str)
                chrom_fmt = f"{chrom_int:02}"
            return f"NC_0000{chrom_fmt}:g.{pos}{ref}>{alt}"

        elif dataset in cdot_datasets:
            hgvs_c = row['hgvs_c']
            ID = row['Ensembl Transcript ID']
            transcript = re.sub(r"(ENST\d+)\.\d+", r"\1", str(ID))
            return f"{transcript}:{hgvs_c}"

        else:
            value = row['aa_alt']
            pos_value = row['aa_pos']
            if pd.notna(pos_value):
                pos = int(pos_value)
            else:
                pos = np.nan
            ID = row['Ensembl Transcript ID']
            aa_ref = row['aa_ref']
            amino_acid = aa_ref if value in ['0', '='] else value

            if pd.notna(ID) and isinstance(ID, str) and amino_acid in amino_acid_to_codon:
                transcript = re.sub(r"(ENST\d+)\.\d+", r"\1", ID)
                codons = amino_acid_to_codon[amino_acid]
                nucleotide_pos = (int(float(pos)) * 3) - 2

                # Return a list of keys to allow codon expansion
                keys = []
                for codon in codons:
                    if amino_acid == '-':
                        raise ValueError("Protein deletion, check manually")
                    else:
                        key = f"{transcript}:c.{nucleotide_pos}_{nucleotide_pos + 2}delins{codon}"
                    keys.append(key)
                return keys

            else:
                raise ValueError("Invalid amino acid data")

    except Exception as e:
        print(f"[WARN] Key generation failed for dataset={dataset}: {e} - {ID}:{aa_ref}{pos}{value}")
        return "placeholder_key"

# Expand rows when generate_key returns a list (codon expansion).
# If a single key is returned, keep one row. If multiple keys are returned,
# duplicate the original row once per key so downstream joins can match any
# codon-consistent nucleotide representation.

def expand_keys(row):
    keys = generate_key(row)
    if isinstance(keys, list):
        # Return a DataFrame with one row per key
        return pd.DataFrame([row.to_dict() | {'key': k} for k in keys])
    else:
        # Return a DataFrame with a single row
        return pd.DataFrame([row.to_dict() | {'key': keys}])

expanded_df = pd.concat(
    pd_1.apply(expand_keys, axis=1).to_list(),
    ignore_index=True
)

# Save to CSV
expanded_df.to_csv(TMP_DIR / "supp_data_combined_curation_key.csv.gz", compression = 'gzip',index=False)
print(f"[OK] Saved expanded dataframe with generated keys: {expanded_df.shape}")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/4271367648.py:41: DtypeWarning: Columns (3,7,8,9,10,11,12,13,16,18,19,20,21,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_1 = pd.read_csv(TMP_DIR / "supp_data_combined_curation.csv.gz", header=0)


[WARN] Key generation failed for dataset=PTEN_Matreyek_2018: Invalid amino acid data - ENST00000371953.8:nannannan
[WARN] Key generation failed for dataset=SCN5A_Ma_2024: Invalid amino acid data - ENST00000423572.7:nannannan
[WARN] Key generation failed for dataset=G6PD_IGVF: Invalid amino acid data - ENST00000393562.10:nannannan
[WARN] Key generation failed for dataset=G6PD_IGVF: Invalid amino acid data - ENST00000393562.10:nannannan
[OK] Saved expanded dataframe with generated keys: (1097873, 51)


In [172]:
# --- Validation Block: KeyGen Integrity Check ---

# Check unique IDs
ids_before = set(pd_1['ID'].dropna().unique())
ids_after = set(expanded_df['ID'].dropna().unique())

if ids_before != ids_after:
    missing_ids = ids_before - ids_after
    new_ids = ids_after - ids_before
    raise ValueError(
        f"[ERROR] Merge altered unique IDs.\n"
        f"Missing IDs after merge: {missing_ids}\n"
        f"New unexpected IDs after merge: {new_ids}\n"
        f"Count before: {len(ids_before)}, after: {len(ids_after)}"
    )
else:
    print(f"[OK] Unique IDs preserved across merge (n={len(ids_before)}).")


[OK] Unique IDs preserved across merge (n=434166).


In [175]:
import pandas as pd

#load pillar project combined data with key gen
pd_1_key = pd.read_csv(TMP_DIR / "supp_data_combined_curation_key.csv.gz")

#strip white spaces and such 
pd_1_key = pd_1_key.applymap(lambda x: x.strip() if isinstance(x, str) else x)

#load combined VEP output
pd_2_key = pd.read_csv(DATA_DIR/"VEP_output/VEP_output.csv.gz")

#strip white spaces and such
pd_2_key = pd_2_key.applymap(lambda x: x.strip() if isinstance(x, str) else x)

pd_2_key = pd_2_key.apply(pd.to_numeric, errors='ignore')

#define datatypes for columns to be merged 
pd_1_key['key'] = pd_1_key['key'].astype(str)
pd_1_key['Ensembl Transcript ID'] = pd_1_key['Ensembl Transcript ID'].astype(str)
pd_2_key['#Uploaded_variation'] = pd_2_key['#Uploaded_variation'].astype(str)
pd_2_key['Feature'] = pd_2_key['Feature'].astype(str)

#filter on datasets with only amino acid information first to carry through for some changes downstream
pd_1_filtered = pd_1_key[pd_1_key['nucleotide_or_aa'] == 'aa']

#merge on key and ensembl transcript ID 
pd_merged = pd.merge(pd_1_filtered, pd_2_key[["Location","Allele","Consequence","HGVSc","HGVSp",
                                          "cDNA_position","CDS_position","Protein_position","Amino_acids",
                                          "Codons","REF_ALLELE","Feature","#Uploaded_variation","UPLOADED_ALLELE",
                                          "STRAND"]], 
                                          left_on=["key",'Ensembl Transcript ID'],
                                          right_on = ["#Uploaded_variation","Feature"],how='left')

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1653368280.py:4: DtypeWarning: Columns (3,7,8,9,10,11,12,13,16,18,19,20,21,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_1_key = pd.read_csv(TMP_DIR / "supp_data_combined_curation_key.csv.gz")
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1653368280.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pd_1_key = pd_1_key.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1653368280.py:10: DtypeWarning: Columns (14,16,23,30) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_2_key = pd.read_csv(DATA_DIR/"VEP_output/VEP_output.csv.gz")
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1653368280.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pd_2_key = pd_2_key.applymap(lambda x: x.strip() if isinstan

In [176]:
#take the merged dataframe and clean it up and drop any duplicates

pd_merged = pd_merged.applymap(lambda x: x.strip() if isinstance(x, str) else x)

pd_merged = pd_merged.apply(pd.to_numeric, errors='ignore')

pd_merged_clean = pd_merged.drop_duplicates()

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1006313645.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pd_merged = pd_merged.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1006313645.py:5: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  pd_merged = pd_merged.apply(pd.to_numeric, errors='ignore')


In [177]:
import re
import numpy as np

# Harmonize ref/alt alleles (strand-aware) and extract hg38 and transcript-level annotations

pd_merged_clean['STRAND'] = pd_merged_clean['STRAND'].astype(float)

nuc_dic = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}

def translate_sequence(sequence):
    return ''.join(nuc_dic.get(nuc, nuc) for nuc in sequence)[::-1]

def process_alleles(row):
    if pd.notna(row['UPLOADED_ALLELE']):
        ref, alt = str(row['UPLOADED_ALLELE']).split("/")
        strand = row.get('STRAND')
        var = row.get('#Uploaded_variation')
        if pd.notna(strand) and strand == -1.0 and var.startswith('ENST'):
            return translate_sequence(ref), translate_sequence(alt)
        return ref, alt
    return np.nan, np.nan

def set_transcript_alleles(row):
    ref, alt = row['ref_allele'], row['alt_allele']
    strand = row.get('STRAND')
    
    if pd.notna(ref) and pd.notna(alt):
        if pd.notna(strand) and strand == -1.0:
            return translate_sequence(ref), translate_sequence(alt)
        else:
            return ref, alt
    return np.nan, np.nan

pd_merged_clean[['ref_allele', 'alt_allele']] = pd_merged_clean.apply(process_alleles, axis=1, result_type="expand")
pd_merged_clean[['transcript_ref', 'transcript_alt']] = pd_merged_clean.apply(set_transcript_alleles, axis=1, result_type="expand")


def extract_hg38_position(value):
    if pd.notna(value):
        chrom, pos = str(value).split(":")
        start, end = pos.split("-")
        return start, end
    return value, value

pd_merged_clean[['hg38_start', 'hg38_end']] = pd_merged_clean['Location'].apply(extract_hg38_position).apply(pd.Series)

pd_merged_clean['hgvs_c'] = pd_merged_clean['HGVSc'].apply(lambda x: x.split(":")[1] if pd.notna(x) and ":" in x else x)

pd_merged_clean['transcript_pos'] = pd_merged_clean['CDS_position']
pd_merged_clean['consequence'] = pd_merged_clean['Consequence']

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/981418197.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_merged_clean['STRAND'] = pd_merged_clean['STRAND'].astype(float)
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/981418197.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_merged_clean[['ref_allele', 'alt_allele']] = pd_merged_clean.apply(process_alleles, axis=1, result_type="expand")
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/981418197.py:35: SettingWithCopyWarning: 
A value is tryin

In [178]:
# Flag amino-acid changes (ref/alt) for inspection to ensure correct mapping 

pd_merged_clean['check_right'] = pd_merged_clean.apply(
    lambda row: row['aa_ref']
    if row['aa_ref'] == row['aa_alt']
    else f"{row['aa_ref']}/{row['aa_alt']}",
    axis=1
)

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/3567033551.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_merged_clean['check_right'] = pd_merged_clean.apply(


In [179]:
import numpy as np

#check to ensure anything is flagged where amino acids don't map

pd_merged_clean['Flag'] = np.where(
    pd_merged_clean['Amino_acids'].notna() & 
    (pd_merged_clean['check_right'] != pd_merged_clean['Amino_acids']),
    '*',
    ''
)

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1115537506.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_merged_clean['Flag'] = np.where(


In [180]:
#add variant validator output 

vv = pd.read_csv(DATA_DIR/"VariantValidator_output/variant_validator_output_1.csv.gz")
vv = vv.drop_duplicates(subset='Input')

def build_variant_validator(row):
    transcript = row.get('Ensembl Transcript ID')
    uploaded = row.get('#Uploaded_variation')

    if pd.isna(transcript) or pd.isna(uploaded):
        return np.nan

    if ':' not in str(uploaded):
        return np.nan

    try:
        variant_part = str(uploaded).split(':')[1]
        return f"{transcript}:{variant_part}"
    except IndexError:
        return np.nan

pd_merged_clean['Variant_validator'] = pd_merged_clean.apply(build_variant_validator, axis=1)

merge = pd.merge(pd_merged_clean, vv, left_on = 'Variant_validator', right_on = 'Input', how = 'left')

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/3447480155.py:3: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  vv = pd.read_csv(DATA_DIR/"VariantValidator_output/variant_validator_output_1.csv.gz")
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/3447480155.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_merged_clean['Variant_validator'] = pd_merged_clean.apply(build_variant_validator, axis=1)


In [181]:
import pandas as pd

#create standardized hgvsp

merge['hgvs_p_clean'] = (
    merge['HGVS_Predicted_Protein']
    .astype(str)
    .str.extract(r'(p\.\s*\(?[A-Za-z]{3}\d+(?:[A-Za-z]{3}|\*|=))')[0]
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)



merge['hgvs_p_clean'] = merge['hgvs_p_clean'].str.replace('Ter', '*', regex=False)
merge['hgvs_p_clean'] = merge['hgvs_p_clean'].str.replace('ext*', '', regex=False)
merge['hgvs_p'] = merge['hgvs_p'].str.replace('Ter', '*', regex=False)
merge['hgvs_p'] = np.where(
    merge['aa_ref'] == merge['aa_alt'],
    merge['hgvs_p'].str.replace(
        r'([A-Za-z]{3}\d{1,5})[A-Za-z]{3}$',  # handles 1–5 digit positions
        r'\1=',
        regex=True
    ),
    merge['hgvs_p']
)

In [182]:
import re

#use variant validator input to update dataframe

def xsm(row):
    if row['Flag'] == '*':
        updated = False  

        # Extract genomic positions from HGVSg
        hgvs_g = row.get('HGVS_Genomic_GRCh38')
        if pd.notna(hgvs_g) and 'g.' in hgvs_g:
            match = re.search(r'g\.(\d+)_?(\d+)?', hgvs_g)
            if match:
                row['hg38_start'] = int(match.group(1))
                row['hg38_end'] = int(match.group(2)) if match.group(2) else int(match.group(1))
                updated = True

        # Extract transcript-level c. position
        hgvs_t = row.get('HGVS_transcript')
        if pd.notna(hgvs_t) and 'c.' in hgvs_t:
            match = re.search(r'c\.(\d+)_?(\d+)?', hgvs_t)
            if match:
                row['transcript_pos'] = f"{match.group(1)}-{match.group(2)}" if match.group(2) else match.group(1)
                updated = True
        
        # Assign remaining values
        if pd.notna(hgvs_t):
            row['hgvs_c'] = hgvs_t.split(':')[1] if ':' in hgvs_t else hgvs_t
            updated = True
        row['ref_allele'] = row['GRCh38_REF']
        row['alt_allele'] = row['GRCh38_ALT']
        row['consequence'] = np.nan

        # Mark Flag as updated if we changed anything
        if updated:
            row['Flag'] = 'updated'
    
    return row

                                                                                  
merge = merge.apply(xsm, axis=1)

In [183]:
#updated flags

merge['Flag'] = merge['Flag'] = np.where(
    merge['hgvs_p_clean'].notna() & 
    (merge['hgvs_p'] != merge['hgvs_p_clean']),
    '*',
    merge['Flag'])

In [184]:
#drop variant validator columns not needed

merge = merge.drop(columns = ['Variant_validator','Input','Warnings','Select transcript',
 'HGVS_transcript',
 'HGVS_intronic_chr_context',
 'HGVS_intronic_rsg_context','HGVS_RefSeqGene',
 'HGVS_LRG',
 'HGVS_LRG_transcript',
 'HGVS_Predicted_Protein',
 'HGVS_Genomic_GRCh37',
 'GRCh37_CHR',
 'GRCh37_POS',
 'GRCh37_ID',
 'GRCh37_REF',
 'GRCh37_ALT',
 'HGVS_Genomic_GRCh38',
 'GRCh38_CHR',
 'GRCh38_POS',
 'GRCh38_ID',
 'GRCh38_REF',
 'GRCh38_ALT',
 'Gene_Symbol',
 'HGNC_Gene_ID',
 'Transcript_description',
 'Alt_genomic_loci','hgvs_p_clean'])

In [185]:
#create another key for variant validator 

def build_variant_validator_updated(row):
    transcript = row.get('Ensembl Transcript ID')
    uploaded = row.get('key')

    # Handle missing required fields
    if pd.isna(transcript) or pd.isna(uploaded):
        return "could_not_create_key"

    uploaded_str = str(uploaded)

    # Ensure there's a ":" to split on
    if ':' not in uploaded_str:
        return "could_not_create_key"

    try:
        variant_part = uploaded_str.split(':', 1)[1]  # split only once
        if not variant_part or variant_part.lower() == 'nan':
            return "could_not_create_key"
        return f"{transcript}:{variant_part}"
    except Exception:
        return "could_not_create_key"

mask_x = (
    merge['hg38_start'].isna() &
    merge['Location'].isna()
)

merge['Variant_validator'] = pd.Series(dtype="string") 

merge.loc[mask_x, 'Variant_validator'] = (
    merge.loc[mask_x].apply(build_variant_validator_updated, axis=1)
)

change = merge['Variant_validator'].notna()

merge.loc[change, 'Variant_validator'] = (
    merge.loc[change, 'Variant_validator']
    .str.replace(r'delinsdel$', 'del', regex=True)
)

In [186]:
#load round 2 of variant validator 

df = pd.read_csv(DATA_DIR/"VariantValidator_output/variant_validator_output_2.csv.gz")

df = df.drop_duplicates(subset = 'Input')

import pandas as pd
merge3 = pd.merge(merge, df, left_on = 'Variant_validator', right_on = 'Input', how = 'left')

In [187]:
#creating standarized hgvsp column for use

merge3['hgvs_p_clean'] = (
    merge3['HGVS_Predicted_Protein']
    .astype(str)
    .str.extract(r'(p\.\s*\(?[A-Za-z]{3}\d+(?:[A-Za-z]{3}|\*|=))')[0]
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)

merge3['hgvs_p_clean'] = merge3['hgvs_p_clean'].str.replace('Ter', '*', regex=False)
merge3['hgvs_p_clean'] = merge3['hgvs_p_clean'].str.replace('ext*', '', regex=False)
merge3['hgvs_p'] = np.where(
    merge3['aa_ref'] == merge3['aa_alt'],
    merge3['hgvs_p'].str.replace(
        r'([A-Za-z]{3}\d{1,5})[A-Za-z]{3}$', 
        r'\1=',
        regex=True
    ),
    merge3['hgvs_p']
)

In [188]:
import numpy as np
import re
import pandas as pd

#set information from variant validator files to the dataframe

def xsm_input(row):
    updated = False  # Track whether we updated anything

    if pd.notna(row.get('Input')):

        # Extract genomic positions from HGVSg
        hgvs_g = row.get('HGVS_Genomic_GRCh38')
        if pd.notna(hgvs_g) and 'g.' in hgvs_g:
            match = re.search(r'g\.(\d+)_?(\d+)?', hgvs_g)
            if match:
                new_start = int(match.group(1))
                new_end = int(match.group(2)) if match.group(2) else new_start
                if row.get('hg38_start') != new_start or row.get('hg38_end') != new_end:
                    row['hg38_start'] = new_start
                    row['hg38_end'] = new_end
                    updated = True

        # Extract transcript-level position from HGVS_transcript
        hgvs_t = row.get('HGVS_transcript')
        if pd.notna(hgvs_t) and 'c.' in hgvs_t:
            match = re.search(r'c\.(\d+)_?(\d+)?', hgvs_t)
            if match:
                new_transcript_pos = (
                    f"{match.group(1)}-{match.group(2)}" if match.group(2) else match.group(1)
                )
                if row.get('transcript_pos') != new_transcript_pos:
                    row['transcript_pos'] = new_transcript_pos
                    updated = True

        # Assign c.HGVS string
        if pd.notna(hgvs_t):
            new_hgvs_c = hgvs_t.split(':')[1] if ':' in hgvs_t else hgvs_t
            if row.get('hgvs_c') != new_hgvs_c:
                row['hgvs_c'] = new_hgvs_c
                updated = True

        # Always set alleles (optional: you can wrap these in checks too)
        row['ref_allele'] = row.get('GRCh38_REF')
        row['alt_allele'] = row.get('GRCh38_ALT')

        # Set consequence to NaN if it wasn’t already
        if pd.isna(row.get('consequence')):
            row['consequence'] = np.nan

        # Set flag if anything was updated
        if updated:
            row['Flag'] = 'updated_2'

    return row

# Apply function
merge3 = merge3.apply(xsm_input, axis=1)


In [189]:
#add flags where hgvsp do not equal each other

merge3['Flag'] = merge3['Flag'] = np.where(
    merge3['hgvs_p_clean'].notna() & 
    (merge3['hgvs_p'] != merge3['hgvs_p_clean']),
    '*',
    merge3['Flag'])

In [190]:
# Fill missing STRAND values using the gene-level strand annotation

strand_map = (
    merge3.groupby('Gene')['STRAND']
          .first()  
)

merge3['STRAND'] = merge3.apply(
    lambda row: strand_map[row['Gene']] if pd.isna(row['STRAND']) else row['STRAND'],
    axis=1
)

In [191]:
#set transcript alleles given strand orientation of the gene

def set_transcript_alleles(row):
    ref, alt = row['ref_allele'], row['alt_allele']
    strand = row.get('STRAND')
    
    if pd.notna(ref) and pd.notna(alt):
        if pd.notna(strand) and strand == -1.0:
            return translate_sequence(ref), translate_sequence(alt)
        else:
            return ref, alt
    return np.nan, np.nan

merge3['STRAND'] = merge3['STRAND'].astype(float)

# Build mask
strand_mask = merge3['Input'].notna()

# Apply with reindexing to avoid alignment issues
new_vals = (
    merge3.loc[strand_mask]
    .apply(set_transcript_alleles, axis=1, result_type="expand")
)
merge3.loc[strand_mask, ['transcript_ref', 'transcript_alt']] = new_vals.values

In [192]:
#drop variant validator columns 

merge3.drop(columns=['Location', 'Allele', 'Consequence', 'HGVSc', 'HGVSp', 'cDNA_position', 'CDS_position', 
                       'Protein_position', 'Amino_acids', 'Codons', 'REF_ALLELE', 'Feature', '#Uploaded_variation', 'UPLOADED_ALLELE', 
                    'check_right', 'Variant_validator', 'Input', 'Warnings', 'Select transcript', 'HGVS_transcript', 
                       'HGVS_intronic_chr_context', 'HGVS_intronic_rsg_context', 'HGVS_RefSeqGene', 'HGVS_LRG', 'HGVS_LRG_transcript', 
                       'HGVS_Predicted_Protein', 'HGVS_Genomic_GRCh37', 'GRCh37_CHR', 'GRCh37_POS', 'GRCh37_ID', 'GRCh37_REF', 
                       'GRCh37_ALT', 'HGVS_Genomic_GRCh38', 'GRCh38_CHR', 'GRCh38_POS', 'GRCh38_ID', 'GRCh38_REF', 'GRCh38_ALT', 
                       'Gene_Symbol', 'HGNC_Gene_ID', 'Transcript_description', 'Alt_genomic_loci'], inplace=True)
#redo column order
new_column_order = ['ID','key','Dataset', 'Gene', 'HGNC_id', 'Chrom','STRAND','hg19_pos', 'hg38_start','hg38_end',
       'ref_allele', 'alt_allele', 'auth_transcript_id', 'transcript_pos',
       'transcript_ref', 'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt',
       'hgvs_c', 'hgvs_p', 'consequence', 'auth_reported_score',
       'auth_reported_rep_score', 'auth_reported_func_class',
       'splice_measure', 'gnomad_MAF', 'clinvar_sig', 'clinvar_star',
       'clinvar_date_last_reviewed', 'nucleotide_or_aa',
       'Ensembl Transcript ID', 'RefSeq Transcript ID', 'Interval 1 Name', 'Interval 1 Range',
                                           'Interval 1 Class', 'Interval 2 Name', 'Interval 2 Range',
                                           'Interval 2 Class', 'Interval 3 Name', 'Interval 3 Range',
                                           'Interval 3 Class', 'Interval 4 Name', 'Interval 4 Range',
                                           'Interval 4 Class', 'Interval 5 Name', 'Interval 5 Range',
                                           'Interval 5 Class', 'Interval 6 Name', 'Interval 6 Range','Interval 6 Class','Flag']

final_df = merge3[new_column_order]

In [193]:
final_df.to_csv(TMP_DIR/"mapped_data_v1.csv.gz", compression = 'gzip',index = False)

In [194]:
import pandas as pd

final_df = pd.read_csv(TMP_DIR/"mapped_data_v1.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2325129327.py:3: DtypeWarning: Columns (5,12,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv(TMP_DIR/"mapped_data_v1.csv.gz")


In [195]:
import pandas as pd
import numpy as np

# Make sure hg38_start is numeric
final_df['hg38_start'] = pd.to_numeric(final_df['hg38_start'], errors='coerce').astype('Int64')
final_df['Chrom'] = final_df['Chrom'].where(final_df['Chrom'].notna(), None).astype(str)

#create spdi key from VEP output for merge

def generate_spdi(row):
    chrom = row['Chrom']
    pos = row['hg38_start']
    ref = row['ref_allele']
    alt = row['alt_allele']

    # Handle missing values
    if pd.isna(chrom) or pd.isna(pos) or pd.isna(ref) or pd.isna(alt):
        return np.nan

    chrom_str = str(chrom).upper()

    if chrom_str == "X":
        chrom_fmt = "NC_000023"
    elif chrom_str == "Y":
        chrom_fmt = "NC_000024"
    elif chrom_str in ["MT", "M"]:
        chrom_fmt = "NC_012920"
    else:
        chrom_fmt = f"NC_0000{int(chrom_str):02}"

    pos0 = int(pos) - 1

    return f"{chrom_fmt}:{pos0}:{ref}:{alt}"

# Apply row by row
final_df['spdi_key'] = final_df.apply(generate_spdi, axis=1)

In [196]:
#load VEP spdi output for more complete mapping
spdi = pd.read_csv(DATA_DIR/'VEP_output/VEP_output_2.csv.gz')

spdi = spdi.drop_duplicates(subset = ['#Uploaded_variation', 'Feature'])

pd.set_option('display.max_columns', None)

js = pd.merge(final_df, spdi, left_on = ['spdi_key', 'Ensembl Transcript ID'], right_on = ['#Uploaded_variation', 'Feature'], 
              how = 'left')

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1301499536.py:2: DtypeWarning: Columns (16,23,30) have mixed types. Specify dtype option on import or set low_memory=False.
  spdi = pd.read_csv(DATA_DIR/'VEP_output/VEP_output_2.csv.gz')


In [197]:
js.loc[js['consequence'].isna() & js['Consequence'].notna(), 'consequence'] = js['Consequence']

In [198]:
#drop columns from vep output that are not needed

js.drop(columns = ['spdi_key', '#Uploaded_variation', 'Location', 'Allele', 'Consequence', 'IMPACT', 'SYMBOL', 
         'Gene_y', 'Feature_type', 'Feature', 'BIOTYPE', 'EXON', 'INTRON', 'HGVSc', 'HGVSp', 'cDNA_position', 
         'CDS_position', 'Protein_position', 'Amino_acids', 'Codons', 'Existing_variation', 'REF_ALLELE', 'UPLOADED_ALLELE', 
         'DISTANCE', 'STRAND_y', 'FLAGS', 'SYMBOL_SOURCE', 'HGNC_ID', 'MANE', 'MANE_SELECT', 'MANE_PLUS_CLINICAL', 'TSL', 
         'APPRIS', 'SIFT', 'PolyPhen', 'HGVS_OFFSET', 'AF', 'CLIN_SIG', 'SOMATIC', 'PHENO', 'PUBMED', 'MOTIF_NAME', 'MOTIF_POS',
         'HIGH_INF_POS', 'MOTIF_SCORE_CHANGE', 'TRANSCRIPTION_FACTORS'], inplace = True)

js = js.rename(columns=lambda c: c[:-2] if c.endswith('_x') else c)

new_column_order = ['ID','Dataset', 'Gene', 'HGNC_id', 'Chrom','STRAND','hg19_pos', 'hg38_start','hg38_end',
       'ref_allele', 'alt_allele', 'auth_transcript_id', 'transcript_pos',
       'transcript_ref', 'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt',
       'hgvs_c', 'hgvs_p', 'consequence', 'auth_reported_score',
       'auth_reported_rep_score', 'auth_reported_func_class',
       'splice_measure', 'gnomad_MAF', 'clinvar_sig', 'clinvar_star',
       'clinvar_date_last_reviewed', 'nucleotide_or_aa',
       'Ensembl Transcript ID', 'RefSeq Transcript ID', 'Interval 1 Name', 'Interval 1 Range',
                                           'Interval 1 Class', 'Interval 2 Name', 'Interval 2 Range',
                                           'Interval 2 Class', 'Interval 3 Name', 'Interval 3 Range',
                                           'Interval 3 Class', 'Interval 4 Name', 'Interval 4 Range',
                                           'Interval 4 Class', 'Interval 5 Name', 'Interval 5 Range',
                                           'Interval 5 Class', 'Interval 6 Name', 'Interval 6 Range','Interval 6 Class','Flag']

final_df_2 = js[new_column_order]

In [199]:
import numpy as np

#normalize columns and calculate missing values 

nuc_dic = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}

def translate_sequence(sequence):
    return ''.join(nuc_dic.get(nuc, nuc) for nuc in sequence)[::-1]

def process_variants(df):
    # 1. If hg38_end is missing, calculate it
    df.loc[df['hg38_end'].isna(), 'hg38_end'] = (
        df.loc[df['hg38_end'].isna(), 'hg38_start'] +
        df.loc[df['hg38_end'].isna(), 'ref_allele'].str.len() - 1
    )

    # 2. Set transcript_ref and transcript_alt based on strand
    def set_transcripts(row):
        if pd.isna(row['ref_allele']) or pd.isna(row['alt_allele']):
            return pd.Series([np.nan, np.nan])
        if row['STRAND'] == 1.0:
            return pd.Series([row['ref_allele'], row['alt_allele']])
        else:
            return pd.Series([
                translate_sequence(row['ref_allele']),
                translate_sequence(row['alt_allele'])
            ])

    df[['transcript_ref', 'transcript_alt']] = df.apply(set_transcripts, axis=1)

    # 3. Replace '-' with NaN in important columns
    cols = [
        'Gene', 'HGNC_id', 'Chrom',
        'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele',
        'transcript_pos', 'transcript_ref', 'transcript_alt',
        'aa_pos', 'aa_ref', 'aa_alt',
        'hgvs_c', 'hgvs_p', 'consequence'
    ]
    for col in cols:
        df[col] = df[col].replace(['-','---'], np.nan)

    return df

final_df_2 = process_variants(final_df_2)


/tmp/7554300.1.fowler-login.q/ipykernel_1384509/478744052.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df['hg38_end'].isna(), 'hg38_end'] = (
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/478744052.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[['transcript_ref', 'transcript_alt']] = df.apply(set_transcripts, axis=1)
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/478744052.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value i

In [200]:
import pandas as pd
import numpy as np

#drop unmapped expanded variants if the group has atleast one mapping 

def drop_all_unmapped_if_group_has_mapping(df):
    df = df.copy()

    # Check per ID: does the group have any mapped rows based on hg38_start?
    has_mapping = df.groupby('ID').apply(
        lambda g: g['hg38_start'].notna().any()
    ).to_dict()

    # Mask for unmapped rows
    mask_unmapped = df['hg38_start'].isna()

    # Drop unmapped rows if their ID has a mapped entry
    df = df[~(mask_unmapped & df['ID'].map(has_mapping))]

    # Now handle Flag == '*'
    def drop_flag_star(group):
        mapped_non_star = group[
            group['hg38_start'].notna() & (group['Flag'] != '*')
        ]
        if not mapped_non_star.empty:
            return group[group['Flag'] != '*']
        return group

    df = df.groupby('ID', group_keys=False).apply(drop_flag_star)

    return df

final_df_2 = drop_all_unmapped_if_group_has_mapping(final_df_2)

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/4209020479.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  has_mapping = df.groupby('ID').apply(
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/4209020479.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('ID', group_keys=False).apply(drop_flag_star)


In [201]:
#some HMBS variants need to be flagged 

final_df_2.loc[
    (final_df_2['ref_allele'].notna()) & (final_df_2['alt_allele'].isna()), 
    'Flag'
] = '*'

In [202]:
import pandas as pd 

def qc_count_missing(df):
    # Columns we care about
    cols_to_check = [
        'Gene', 'HGNC_id', 'Chrom',
        'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele',
        'transcript_pos', 'transcript_ref', 'transcript_alt',
        'aa_pos', 'aa_ref', 'aa_alt',
        'hgvs_c', 'hgvs_p', 'consequence'
    ]
    
    # Build a summary dict: counts of missing values
    missing_counts = {
        col: df[col].isna().sum()
        for col in cols_to_check
        if col in df.columns
    }
    
    # Convert to DataFrame for a nice report
    report = pd.DataFrame.from_dict(missing_counts, orient='index', columns=['Missing_Count'])
    report = report[report['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)
    
    print("\n[QC REPORT] Count of missing values per column:")
    print(report)
    
    return report

qc_count_missing(final_df_2)


[QC REPORT] Count of missing values per column:
                Missing_Count
consequence             50776
hgvs_c                   1733
transcript_ref            120
alt_allele                120
ref_allele                120
transcript_alt            120
transcript_pos             14
hg38_end                    6
hg38_start                  6
aa_ref                      4
aa_pos                      4
aa_alt                      4
hgvs_p                      1


,Missing_Count
consequence,50776
hgvs_c,1733
transcript_ref,120
alt_allele,120
ref_allele,120
transcript_alt,120
transcript_pos,14
hg38_end,6
hg38_start,6
aa_ref,4


In [203]:
final_df_2.to_csv(TMP_DIR/"mapped_data_v1_5.csv.gz", compression = 'gzip', index = False)

In [204]:
final_df_2 = pd.read_csv(TMP_DIR/"mapped_data_v1_5.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/838524354.py:1: DtypeWarning: Columns (4,11,22,23,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,50) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df_2 = pd.read_csv(TMP_DIR/"mapped_data_v1_5.csv.gz")


In [205]:
import pandas as pd

pd_1 = pd.read_csv(TMP_DIR/"supp_data_combined_curation_key.csv.gz", header=0)

pd_1.drop(columns=['Dataset Name'], inplace=True)

pd_1_nuc = pd_1[pd_1['nucleotide_or_aa'] == 'nucleotide']

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/4193522594.py:3: DtypeWarning: Columns (3,7,8,9,10,11,12,13,16,18,19,20,21,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_1 = pd.read_csv(TMP_DIR/"supp_data_combined_curation_key.csv.gz", header=0)


In [206]:
import pandas as pd

# #nucleotide datasets need some special adjustments to get everything in the right format 

# #BRCA1_2018_Findlay needs hg38 positions annotated

import pandas as pd
from pyliftover import LiftOver

lift_file = DATA_DIR/"Liftover_files/hg19ToHg38.over.chain.gz"

lo = LiftOver(str(lift_file))

# Define a function to lift over positions
def lift_over_pos(row):
    chrom = f"chr{row['Chrom']}"
    pos = row['hg19_pos']
    result = lo.convert_coordinate(chrom, pos)
    if result:
        return result[0][1] 
    else:
        return None 

pd_1_nuc['hg38_start'] = pd_1_nuc.apply(
    lambda row: lift_over_pos(row) if row['Dataset'] == 'BRCA1_Findlay_2018' else row['hg38_start'],
    axis=1
)


/tmp/7554300.1.fowler-login.q/ipykernel_1384509/3610344549.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_1_nuc['hg38_start'] = pd_1_nuc.apply(


In [207]:
import pandas as pd
pd_2_key = pd.read_csv(DATA_DIR/"VEP_output/VEP_output.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/715563619.py:2: DtypeWarning: Columns (14,16,23,30) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_2_key = pd.read_csv(DATA_DIR/"VEP_output/VEP_output.csv.gz")


In [208]:
symbol_strand = pd_2_key[['SYMBOL', 'STRAND']]

In [209]:
symbol_strand = symbol_strand.drop_duplicates(subset = 'SYMBOL')

In [210]:
pd_1_nuc = pd.merge(pd_1_nuc, symbol_strand, left_on = 'Gene', right_on = 'SYMBOL', how = 'left')

In [211]:
import pandas as pd

#add in relevant information from VEP to nucleotide level datasets

pd_2_key['STRAND'] = pd.to_numeric(pd_2_key['STRAND'], errors='coerce')

nuc_dic = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}

def translate_sequence(sequence):
    return ''.join(nuc_dic.get(nuc, nuc) for nuc in sequence)[::-1]

def process_alleles(row):
    if pd.notna(row['UPLOADED_ALLELE']):
        ref, alt = str(row['UPLOADED_ALLELE']).split("/")
        strand = row.get('STRAND')
        var = row.get('#Uploaded_variation')
        if pd.notna(strand) and strand == -1.0 and var.startswith('ENST') :
            return translate_sequence(ref), translate_sequence(alt)
        return ref, alt
    return np.nan, np.nan

def set_transcript_alleles(row):
    ref, alt = row['ref_allele'], row['alt_allele']
    strand = row.get('STRAND')
    
    if pd.notna(ref) and pd.notna(alt):
        if pd.notna(strand) and strand == -1.0:
            return translate_sequence(ref), translate_sequence(alt)
        else:
            return ref, alt
    return np.nan, np.nan

pd_2_key[['ref_allele', 'alt_allele']] = pd_2_key.apply(process_alleles, axis=1, result_type="expand")
pd_2_key[['transcript_ref', 'transcript_alt']] = pd_2_key.apply(set_transcript_alleles, axis=1, result_type="expand")


pd_2_key[['hg38_start', 'hg38_end']] = pd_2_key['Location'].str.extract(r':(\d+)-(\d+)')
pd_2_key['hgvs_c'] = pd_2_key['HGVSc'].str.extract(r':(.*)')
pd_2_key['transcript_pos'] = pd_2_key['HGVSc'].str.extract(r'c\.([\d+-]+)')
pd_2_key['hgvs_p'] = pd_2_key['HGVSp'].str.extract(r':(p\.\w+\d+\w+)')
pd_2_key['aa_pos'] = pd_2_key['Protein_position']
pd_2_key[['aa_ref', 'aa_alt']] = pd_2_key['Amino_acids'].str.split('/', expand=True)
pd_2_key['consequence'] = pd_2_key['Consequence']

In [212]:
pd_key = pd_2_key.drop_duplicates()

In [213]:
pd_key_last = pd_2_key.drop_duplicates(subset = ['hg38_start','ref_allele','alt_allele','Feature'])

In [214]:
#fix BRCA2_Sahu_2025_SGE before merge, accidentally carried over the version number for NC format. its also needs a special key 

BRCA2_Sahu_2025 = pd.read_excel(DATA_DIR/"Variant_score_files/BRCA2_Sahu_2025_SGE.xlsx", header = 1)
BRCA2_Sahu_2025['c_nom'] = BRCA2_Sahu_2025['c.nom'].str.extract(r'(c\..*)')


# # Create the mapping from c_nom to g.nom
c_to_gnom = BRCA2_Sahu_2025.set_index('c_nom')['g.nom']

# Define the mask for the specific dataset
mask = pd_1_nuc['Dataset'] == 'BRCA2_Sahu_2025_SGE'

# Perform the update only where the mask is True
pd_1_nuc.loc[mask, 'key'] = (
    pd_1_nuc.loc[mask, 'hgvs_c']
    .map(c_to_gnom)
    .combine_first(pd_1_nuc.loc[mask, 'key'])  # fallback to original if no match
)


In [215]:
mer = pd.merge(pd_1_nuc, pd_key, left_on = ['key','Ensembl Transcript ID'], right_on = ['#Uploaded_variation','Feature'], how = 'left')

In [216]:
for col in mer.columns:
    if col.endswith('_x') and col[:-2] + '_y' in mer.columns:
        mer[col] = mer[col].fillna(mer[col[:-2] + '_y'])

In [217]:
#Sahu needs some special help, ref and alt are messed up 

pd.set_option('display.max_columns', None)

mask_Sahu = mer['Dataset'] == 'BRCA2_Sahu_2025_SGE'

mer.loc[mask, 'ref_allele_x'] = mer.loc[mask, 'ref_allele_y'].where(
    mer.loc[mask, 'ref_allele_y'].notna(), mer.loc[mask, 'ref_allele_x']
)

mer.loc[mask, 'alt_allele_x'] = mer.loc[mask, 'alt_allele_y'].where(
    mer.loc[mask, 'alt_allele_y'].notna(), mer.loc[mask, 'alt_allele_x']
)

mer.loc[mask, 'transcript_ref_x'] = mer.loc[mask, 'transcript_ref_y'].where(
    mer.loc[mask, 'transcript_ref_y'].notna(), mer.loc[mask, 'transcript_ref_x']
)

mer.loc[mask, 'transcript_alt_x'] = mer.loc[mask, 'transcript_alt_y'].where(
    mer.loc[mask, 'transcript_alt_y'].notna(), mer.loc[mask, 'transcript_alt_x']
)


In [218]:
#BAP1 needs hgvs_c and hgvs_p normalized to match the dataframe

mer.loc[mer['Dataset'] == 'BAP1_Waters_2024', 'hgvs_c_x'] = mer.loc[
    mer['Dataset'] == 'BAP1_Waters_2024', 'hgvs_c_x'].str.split(':').str[1]

In [219]:
mer = mer.rename(columns={col: col[:-2] for col in mer.columns if col.endswith('_x')})

In [220]:
mer.to_csv(TMP_DIR/"mapped_data_v2.csv.gz", compression = 'gzip', index = False)

In [221]:
import pandas as pd

mer = pd.read_csv(TMP_DIR/"mapped_data_v2.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/443030622.py:3: DtypeWarning: Columns (3,9,10,11,12,13,19,20,31,32,33,34,35,36,37,38,39,40,41,42,50,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,103,104,105,106,107,108,109) have mixed types. Specify dtype option on import or set low_memory=False.
  mer = pd.read_csv(TMP_DIR/"mapped_data_v2.csv.gz")


In [222]:
import pandas as pd
import numpy as np

# Make sure hg38_start is numeric
mer['hg38_start'] = pd.to_numeric(mer['hg38_start'], errors='coerce').astype('Int64')

def generate_spdi(row):
    chrom = row['Chrom']
    pos = row['hg38_start']
    ref = row['ref_allele']
    alt = row['alt_allele']

    # Handle missing values
    if pd.isna(chrom) or pd.isna(pos) or pd.isna(ref) or pd.isna(alt):
        return np.nan

    chrom_str = str(chrom).upper()

    if chrom_str == "X":
        chrom_fmt = "NC_000023"
    elif chrom_str == "Y":
        chrom_fmt = "NC_000024"
    elif chrom_str in ["MT", "M"]:
        chrom_fmt = "NC_012920"
    else:
        chrom_fmt = f"NC_0000{int(chrom_str):02}"

    pos0 = int(pos) - 1

    return f"{chrom_fmt}:{pos0}:{ref}:{alt}"

# Apply row by row
mer['spdi_key'] = mer.apply(generate_spdi, axis=1)

In [223]:
spdi_nuc_2 = pd.read_csv(DATA_DIR/"VEP_output/VEP_output_3.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/511271484.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  spdi_nuc_2 = pd.read_csv(DATA_DIR/"VEP_output/VEP_output_3.csv.gz")


In [224]:
spdi_nuc_2 = spdi_nuc_2.drop_duplicates(subset = ['#Uploaded_variation', 'Feature'])

spdi_nuc_2['hgvs_c'] = spdi_nuc_2['HGVSc'].str.extract(r':(.*)')
spdi_nuc_2['transcript_pos'] = spdi_nuc_2['HGVSc'].str.extract(r'c\.([\d+-]+)')
spdi_nuc_2['hgvs_p'] = spdi_nuc_2['HGVSp'].str.extract(r':(p\.\w+\d+\w+)')
spdi_nuc_2['aa_pos'] = spdi_nuc_2['Protein_position']
spdi_nuc_2[['aa_ref', 'aa_alt']] = spdi_nuc_2['Amino_acids'].str.split('/', expand=True)
spdi_nuc_2['consequence'] = spdi_nuc_2['Consequence']

bn = pd.merge(mer, spdi_nuc_2, left_on = ['spdi_key', 'Ensembl Transcript ID'], right_on = ['#Uploaded_variation', 'Feature'], 
              how = 'left', suffixes = ('','_m'))

In [225]:
cols_to_update = [
    'ref_allele', 'alt_allele',
    'transcript_pos', 'transcript_ref', 'transcript_alt',
    'aa_pos', 'aa_ref', 'aa_alt',
    'hgvs_c', 'hgvs_p', 'consequence'
]

for col in cols_to_update:
    source_col = f"{col}_m"  
    if source_col in bn.columns:
        if col in ['hgvs_c', 'hgvs_p']:
            # Update only if target is NaN, and keep only part after ':'
            bn.loc[bn[col].isna() & bn[source_col].notna(), col] = (
                bn[source_col].str.split(':').str[-1]
            )
        else:
            # Update only if target is NaN
            bn.loc[bn[col].isna() & bn[source_col].notna(), col] = bn[source_col]


In [226]:
import pandas as pd
import numpy as np

bn['STRAND'] = pd.to_numeric(bn['STRAND'], errors='coerce')

nuc_dic = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}

def translate_sequence(sequence):
    return ''.join(nuc_dic.get(nuc, nuc) for nuc in sequence)[::-1]

def process_variants(df):
    # 1. If hg38_end is missing, calculate it
    df.loc[df['hg38_end'].isna(), 'hg38_end'] = (
        df.loc[df['hg38_end'].isna(), 'hg38_start'] +
        df.loc[df['hg38_end'].isna(), 'ref_allele'].str.len() - 1
    )

    # 2. Set transcript_ref and transcript_alt based on strand
    def set_transcripts(row):
        if pd.isna(row['ref_allele']) or pd.isna(row['alt_allele']):
            return pd.Series([np.nan, np.nan])
        if row['STRAND'] == 1.0:
            return pd.Series([row['ref_allele'], row['alt_allele']])
        else:
            return pd.Series([
                translate_sequence(row['ref_allele']),
                translate_sequence(row['alt_allele'])
            ])

    df[['transcript_ref', 'transcript_alt']] = df.apply(set_transcripts, axis=1)

    # 3. Replace '-' with NaN in important columns
    cols = [
        'Gene', 'HGNC_id', 'Chrom',
        'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele',
        'transcript_pos', 'transcript_ref', 'transcript_alt',
        'aa_pos', 'aa_ref', 'aa_alt',
        'hgvs_c', 'hgvs_p', 'consequence'
    ]
    for col in cols:
        df[col] = df[col].replace(['-','---'], np.nan)

    return df

bn = process_variants(bn)

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2858233257.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<FloatingArray>
[214728695.0, 214728702.0, 214728710.0, 214728712.0, 214728713.0, 214728713.0,
 214728714.0, 214728722.0, 214728724.0, 214728724.0,
 ...
  58734182.0,  58734189.0,  58734198.0,  58734201.0,  58734207.0,  58734210.0,
  58734213.0,  58734216.0,  32371081.0,  32371081.0]
Length: 20915, dtype: Float64' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df['hg38_end'].isna(), 'hg38_end'] = (


In [227]:
bn.drop(columns = ['#Uploaded_variation', 'key', 'Location', 'Allele', 'Consequence', 'IMPACT', 'SYMBOL_y', 'Gene_y', 'Feature_type', 
                   'Feature', 'BIOTYPE', 'EXON', 'INTRON', 'HGVSc', 'HGVSp', 'cDNA_position', 'CDS_position', 'Protein_position', 
                   'Amino_acids', 'Codons', 'Existing_variation', 'REF_ALLELE', 'UPLOADED_ALLELE', 'DISTANCE', 'STRAND_y', 'FLAGS', 
                   'SYMBOL_SOURCE', 'HGNC_ID', 'MANE', 'MANE_SELECT', 'MANE_PLUS_CLINICAL', 'TSL', 'APPRIS', 'SIFT', 'PolyPhen', 
                   'HGVS_OFFSET', 'AF', 'CLIN_SIG', 'SOMATIC', 'PHENO', 'PUBMED', 'MOTIF_NAME', 'MOTIF_POS', 'HIGH_INF_POS', 
                   'MOTIF_SCORE_CHANGE', 'TRANSCRIPTION_FACTORS', 'ref_allele_y', 'alt_allele_y', 'transcript_ref_y', 'transcript_alt_y', 
                   'hg38_start_y', 'hg38_end_y', 'hgvs_c_y', 'transcript_pos_y', 'hgvs_p_y', 'aa_pos_y', 'aa_ref_y', 'aa_alt_y', 
                   'consequence_y', 'spdi_key', '#Uploaded_variation_m', 'Location_m', 'Allele_m', 
                   'Consequence_m', 'IMPACT_m', 'SYMBOL_m', 'Gene_m', 'Feature_type_m', 'Feature_m', 'BIOTYPE_m', 'EXON_m', 
                   'INTRON_m', 'HGVSc_m', 'HGVSp_m', 'cDNA_position_m', 'CDS_position_m', 'Protein_position_m', 'Amino_acids_m', 
                   'Codons_m', 'Existing_variation_m', 'REF_ALLELE_m', 'UPLOADED_ALLELE_m', 'DISTANCE_m', 'STRAND_m', 'FLAGS_m',
                   'SYMBOL_SOURCE_m', 'HGNC_ID_m', 'MANE_m', 'MANE_SELECT_m', 'MANE_PLUS_CLINICAL_m', 'TSL_m', 'APPRIS_m', 
                   'SIFT_m', 'PolyPhen_m', 'HGVS_OFFSET_m', 'AF_m', 'CLIN_SIG_m', 'SOMATIC_m', 'PHENO_m', 'PUBMED_m', 'MOTIF_NAME_m',
                   'MOTIF_POS_m', 'HIGH_INF_POS_m', 'MOTIF_SCORE_CHANGE_m', 'TRANSCRIPTION_FACTORS_m', 'hgvs_c_m', 'transcript_pos_m', 
                   'hgvs_p_m', 'aa_pos_m', 'aa_ref_m', 'aa_alt_m', 'consequence_m'], inplace = True)

bn = bn.rename(columns=lambda c: c[:-2] if c.endswith('_x') else c)

new_column_order = ['ID','Dataset', 'Gene', 'HGNC_id', 'Chrom','STRAND','hg19_pos', 'hg38_start','hg38_end',
       'ref_allele', 'alt_allele', 'auth_transcript_id', 'transcript_pos',
       'transcript_ref', 'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt',
       'hgvs_c', 'hgvs_p', 'consequence', 'auth_reported_score',
       'auth_reported_rep_score', 'auth_reported_func_class',
       'splice_measure', 'gnomad_MAF', 'clinvar_sig', 'clinvar_star',
       'clinvar_date_last_reviewed', 'nucleotide_or_aa',
       'Ensembl Transcript ID', 'RefSeq Transcript ID', 'Interval 1 Name', 'Interval 1 Range',
                                           'Interval 1 Class', 'Interval 2 Name', 'Interval 2 Range',
                                           'Interval 2 Class', 'Interval 3 Name', 'Interval 3 Range',
                                           'Interval 3 Class', 'Interval 4 Name', 'Interval 4 Range',
                                           'Interval 4 Class', 'Interval 5 Name', 'Interval 5 Range',
                                           'Interval 5 Class', 'Interval 6 Name', 'Interval 6 Range','Interval 6 Class']

bn = bn[new_column_order]

In [228]:
bn.to_csv(TMP_DIR/"mapped_data_v2_5.csv.gz", compression = 'gzip',index = False)

In [229]:
import pandas as pd
bn = pd.read_csv(TMP_DIR/"mapped_data_v2_5.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2842598550.py:2: DtypeWarning: Columns (4,11,15,21,22,32,33,34,35,36,37,38,39,40,41,42,43) have mixed types. Specify dtype option on import or set low_memory=False.
  bn = pd.read_csv(TMP_DIR/"mapped_data_v2_5.csv.gz")


In [230]:
final = pd.concat([final_df_2, bn], axis=0, join='outer', ignore_index=True)

In [231]:
#simplified consequence 

# load extended version of the Ensembl consequence table
# original table extracted from the website https://asia.ensembl.org/info/genome/variation/prediction/predicted_data.html on 2025-03-18
consequence_df = pd.read_csv(DATA_DIR/"Simplified_consequence_file/extended_ensembl_consequence.csv.gz")

#Create a dictionary mapping terms to their SO summary term (prioritizing importance)
mapping_dict = (
    consequence_df.sort_values(by="Importance", ascending=False)
    .groupby("VEP output term")[["SO summary term", "Importance"]]
    .first()
    .to_dict()["SO summary term"]
)

extra_terms = ['UTR_variant', 'splicing_variant', 'splice_site_variant']

mapping_dict.update({term: term for term in extra_terms})


#Function to assign the most important SO summary term
def get_simplified_consequence(vep_terms):
    if not isinstance(vep_terms, str):  
        return np.nan
    terms = vep_terms.split(',')  
    matched_terms = [mapping_dict.get(term.strip()) for term in terms if term.strip() in mapping_dict]  # Match terms
    
    return matched_terms[0] if matched_terms else np.nan

#Apply function to the column
final["simplified_consequence"] = final["consequence"].apply(get_simplified_consequence)

In [232]:
import numpy as np

for col in ['aa_ref', 'aa_alt']:
    final.loc[final[col] == 'STOP', col] = '*'

mask_aa = final['aa_alt'].isna() & (final['simplified_consequence'] == 'synonymous_variant')
final.loc[mask_aa, 'aa_alt'] = final.loc[mask_aa, 'aa_ref']




In [233]:
#filter for SNVs for KCNQ4, extra var output because VEP was wrong earlier
#77 rows that do have SNVs possible are not mapped correctly in VEP using original input, new input file created to merge on.
#these are the amino acids positions associated with that: 105, 178, 377, 431, 505, 538, 582

KCNQ4_vep_exta_var = pd.read_csv(DATA_DIR/"VEP_output/KCNQ4_extra_var_vep_output.csv.gz") 

In [234]:
KCNQ4_vep_exta_var_tx = KCNQ4_vep_exta_var[KCNQ4_vep_exta_var['Feature'] == 'ENST00000347132.10']

KCNQ4_vep_exta_var_tx[['hg38_start', 'hg38_end']] = KCNQ4_vep_exta_var_tx['Location'].str.extract(r'(\d+):(\d+)-(\d+)')[[1, 2]].astype(float).astype('Int64')
KCNQ4_vep_exta_var_tx['ref_allele'] = KCNQ4_vep_exta_var_tx['REF_ALLELE']
KCNQ4_vep_exta_var_tx['transcript_ref'] = KCNQ4_vep_exta_var_tx['REF_ALLELE']
KCNQ4_vep_exta_var_tx['alt_allele'] = KCNQ4_vep_exta_var_tx['Allele']
KCNQ4_vep_exta_var_tx['transcript_alt'] = KCNQ4_vep_exta_var_tx['Allele']
KCNQ4_vep_exta_var_tx['consequence'] = KCNQ4_vep_exta_var_tx['Consequence']
KCNQ4_vep_exta_var_tx['hgvs_c'] = KCNQ4_vep_exta_var_tx['HGVSc'].str.extract(r':(.*)')
KCNQ4_vep_exta_var_tx['hgvs_p'] = KCNQ4_vep_exta_var_tx['HGVSp'].str.extract(r'(p\..*)')
KCNQ4_vep_exta_var_tx['transcript_pos'] = KCNQ4_vep_exta_var_tx['CDS_position']

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1190555809.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  KCNQ4_vep_exta_var_tx[['hg38_start', 'hg38_end']] = KCNQ4_vep_exta_var_tx['Location'].str.extract(r'(\d+):(\d+)-(\d+)')[[1, 2]].astype(float).astype('Int64')
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1190555809.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  KCNQ4_vep_exta_var_tx[['hg38_start', 'hg38_end']] = KCNQ4_vep_exta_var_tx['Location'].str.extract(r'(\d+):(\d+)-(\d+)')[[1, 2]].asty

In [235]:
KCNQ4 = final[final['Gene'] == 'KCNQ4']

In [236]:
KCNQ4_snv = pd.merge(KCNQ4, KCNQ4_vep_exta_var_tx, left_on = ['Ensembl Transcript ID', 'Gene', 'hgvs_p'], 
                     right_on = ['Feature', 'SYMBOL', 'hgvs_p'], how = 'left', suffixes = ('','_y'))

In [237]:
cols_to_update = [
    'hg38_start', 'hg38_end',
    'ref_allele', 'alt_allele',
    'transcript_pos', 'transcript_ref', 'transcript_alt',
    'aa_pos', 'aa_ref', 'aa_alt',
    'hgvs_c', 'hgvs_p', 'consequence'
]

for col in cols_to_update:
    source_col = f"{col}_y"
    if source_col in KCNQ4_snv.columns:
        KCNQ4_snv.loc[
            KCNQ4_snv['hg38_start_y'].notna(),
            col
        ] = KCNQ4_snv[source_col]

In [238]:
#filter all SNV's for KCNQ4, amino acid known codons for PTEN_Mighell and TP53_Boettcher

import pandas as pd

def filter_for_SNVs(df):
    df['ref_allele'] = df['ref_allele'].fillna('').astype(str)

    def keep_group(group):
        snvs = group[group['ref_allele'].str.len() == 1]
        if not snvs.empty:
            return snvs.head(1)  # keep only one SNV
        else:
            return group.head(1)  # keep only one non-SNV

    return (
        df.groupby('ID', group_keys=False)
          .apply(keep_group)
          .reset_index(drop=True)
    )

KCNQ4_snv = filter_for_SNVs(KCNQ4_snv)



/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1035513385.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(keep_group)


In [239]:
KCNQ4_snv = KCNQ4_snv.drop(columns = ['#Uploaded_variation', 'Location', 'Allele', 'Consequence', 'IMPACT', 
                          'SYMBOL', 'Gene_y', 'Feature_type', 'Feature', 'BIOTYPE', 'EXON', 'INTRON', 'HGVSc', 
                          'HGVSp', 'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids', 'Codons', 
                          'Existing_variation', 'REF_ALLELE', 'UPLOADED_ALLELE', 'DISTANCE', 'STRAND_y', 'FLAGS', 'SYMBOL_SOURCE', 
                          'HGNC_ID', 'MANE', 'MANE_SELECT', 'MANE_PLUS_CLINICAL', 'TSL', 'APPRIS', 'SIFT', 'PolyPhen', 
                          'HGVS_OFFSET', 'AF', 'CLIN_SIG', 'SOMATIC', 'PHENO', 'PUBMED', 'MOTIF_NAME', 'MOTIF_POS', 
                          'HIGH_INF_POS', 'MOTIF_SCORE_CHANGE', 'TRANSCRIPTION_FACTORS', 'hg38_start_y', 'hg38_end_y', 
                          'ref_allele_y', 'transcript_ref_y', 'alt_allele_y', 'transcript_alt_y', 'consequence_y', 'hgvs_c_y', 
                          'transcript_pos_y'])

In [240]:
final_snv = final[final['Gene'] != 'KCNQ4']

In [241]:
final_snv = pd.concat([final_snv, KCNQ4_snv], axis=0, ignore_index=True)

In [242]:
import pandas as pd
import numpy as np

# Make sure hg38_start is numeric
final_snv['hg38_start'] = pd.to_numeric(final_snv['hg38_start'], errors='coerce').astype('Int64')

def generate_spdi(row):
    chrom = row['Chrom']
    pos = row['hg38_start']
    ref = row['ref_allele']
    alt = row['alt_allele']

    # Handle missing values
    if pd.isna(chrom) or pd.isna(pos) or pd.isna(ref) or pd.isna(alt):
        return np.nan

    chrom_str = str(chrom).upper()

    if chrom_str == "X":
        chrom_fmt = "NC_000023"
    elif chrom_str == "Y":
        chrom_fmt = "NC_000024"
    elif chrom_str in ["MT", "M"]:
        chrom_fmt = "NC_012920"
    else:
        chrom_fmt = f"NC_0000{int(chrom_str):02}"

    pos0 = int(pos) - 1

    return f"{chrom_fmt}:{pos0}:{ref}:{alt}"

# Apply row by row
final_snv['spdi_key'] = final_snv.apply(generate_spdi, axis=1)

In [243]:
col = ['Gene', 'HGNC_id', 'Chrom',
        'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele',
        'transcript_pos', 'transcript_ref', 'transcript_alt',
        'aa_pos', 'aa_ref', 'aa_alt',
        'hgvs_c', 'hgvs_p', 'consequence']

rows_with_na = final_snv[final_snv[col].isna().any(axis=1)]
rows_with_na['spdi_key'].to_csv('last_spdi_check.txt', index = False)

In [244]:
ls = pd.read_csv(DATA_DIR/"VEP_output/VEP_output_4.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1050339017.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  ls = pd.read_csv(DATA_DIR/"VEP_output/VEP_output_4.csv.gz")


In [245]:
ls = ls.drop_duplicates(subset = ['#Uploaded_variation', 'Feature'])

ls['hgvs_c'] = ls['HGVSc'].str.extract(r':(.*)')
ls['transcript_pos'] = ls['HGVSc'].str.extract(r'c\.([\d+-]+)')
ls['hgvs_p'] = ls['HGVSp'].str.extract(r':(p\.\w+\d+\w+)')
ls['aa_pos'] = ls['Protein_position']
ls[['aa_ref', 'aa_alt']] = ls['Amino_acids'].str.split('/', expand=True)
ls['consequence'] = ls['Consequence']

final_snv_2 = pd.merge(final_snv, ls, left_on = ['spdi_key', 'Ensembl Transcript ID'], right_on = ['#Uploaded_variation', 'Feature'], 
              how = 'left', suffixes = ('','_m'))

In [246]:
cols_to_update_2 = [
    'ref_allele', 'alt_allele',
    'transcript_pos', 'transcript_ref', 'transcript_alt',
    'aa_pos', 'aa_ref', 'aa_alt',
    'hgvs_c', 'hgvs_p', 'consequence'
]

for col in cols_to_update_2:
    source_col = f"{col}_m"  
    if source_col in final_snv_2.columns:
        if col in ['hgvs_c', 'hgvs_p']:
            # Update only if target is NaN, and keep only part after ':'
            final_snv_2.loc[final_snv_2[col].isna() & final_snv_2[source_col].notna(), col] = (
                final_snv_2[source_col].str.split(':').str[-1]
            )
        else:
            # Update only if target is NaN
            final_snv_2.loc[final_snv_2[col].isna() & final_snv_2[source_col].notna(), col] = final_snv_2[source_col]

In [247]:
import numpy as np

nuc_dic = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}

def translate_sequence(sequence):
    return ''.join(nuc_dic.get(nuc, nuc) for nuc in sequence)[::-1]

def process_variants(df):
    # 1. If hg38_end is missing, calculate it
    df.loc[df['hg38_end'].isna(), 'hg38_end'] = (
        df.loc[df['hg38_end'].isna(), 'hg38_start'] +
        df.loc[df['hg38_end'].isna(), 'ref_allele'].str.len() - 1
    )

    # 2. Set transcript_ref and transcript_alt based on strand
    def get_transcript_ref(row):
        if pd.isna(row['ref_allele']):
            return np.nan
        if row['STRAND'] == 1.0:
            return row['ref_allele']
        else:
            return translate_sequence(row['ref_allele'])

    def get_transcript_alt(row):
        if pd.isna(row['alt_allele']):
            return np.nan
        if row['STRAND'] == 1.0:
            return row['alt_allele']
        else:
            return translate_sequence(row['alt_allele'])
    

    df['transcript_ref'] = df.apply(get_transcript_ref, axis=1)
    df['transcript_alt'] = df.apply(get_transcript_alt, axis=1)

    # 3. Replace '-' with NaN in important columns
    cols = [
        'Gene', 'HGNC_id', 'Chrom',
        'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele',
        'transcript_pos', 'transcript_ref', 'transcript_alt',
        'aa_pos', 'aa_ref', 'aa_alt',
        'hgvs_c', 'hgvs_p', 'consequence'
    ]
    for col in cols:
        df[col] = df[col].replace(['-','---'], np.nan)

    return df

final_snv_2 = process_variants(final_snv_2)

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1839765813.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan nan nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df['hg38_end'].isna(), 'hg38_end'] = (
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1839765813.py:45: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace(['-','---'], np.nan)


In [248]:
#interestingly some CHEK2 variants have the wrong ref and alt from VEP at the location for synonymous variants, will need to Flag them 

mask_chk = (final_snv_2['Gene'] == 'CHEK2') & (final_snv_2['hgvs_c'].isna())

chek2_nan_hgvs_rows = final_snv_2.loc[mask_chk]

final_snv_2.loc[chek2_nan_hgvs_rows.index, 'Flag'] = '*'

In [249]:
#BRCA2 Sahu exon 13 variants, three of them need a Flag, p.Gly3086Cys and p.Thr2331Lys

mask_sahu = ((final_snv_2['ID'] == 'BRCA2_Sahu_2023_exon13_SGE_var108241') | (final_snv_2['ID'] == 'BRCA2_Sahu_2023_exon13_Cisplatin_Resistance_var108493') |
             (final_snv_2['ID'] == 'BRCA2_Sahu_2023_exon13_Olaparib_Resistance_var108745')|(final_snv_2['ID'] == 'BRCA2_Sahu_2025_SGE_var308573')|
            (final_snv_2['ID'] == 'BRCA2_Sahu_2023_exon13_global_score_var353620'))

sahu_rows = final_snv_2.loc[mask_sahu]

final_snv_2.loc[sahu_rows.index, 'Flag'] = '*'


In [250]:
#re-run simplified consequence here

#simplified consequence 

# load Alan's extended version of the Ensembl consequence table
# original table extracted from the website https://asia.ensembl.org/info/genome/variation/prediction/predicted_data.html on 2025-03-18
consequence_df = pd.read_csv(DATA_DIR/"Simplified_consequence_file/extended_ensembl_consequence.csv.gz")

#Create a dictionary mapping terms to their SO summary term (prioritizing importance)
mapping_dict = (
    consequence_df.sort_values(by="Importance", ascending=False)
    .groupby("VEP output term")[["SO summary term", "Importance"]]
    .first()
    .to_dict()["SO summary term"]
)

extra_terms = ['UTR_variant', 'splicing_variant', 'splice_site_variant']

mapping_dict.update({term: term for term in extra_terms})


#Function to assign the most important SO summary term
def get_simplified_consequence(vep_terms):
    if not isinstance(vep_terms, str):  
        return np.nan
    terms = vep_terms.split(',')  
    matched_terms = [mapping_dict.get(term.strip()) for term in terms if term.strip() in mapping_dict]  # Match terms
    
    return matched_terms[0] if matched_terms else np.nan

#Apply function to the column
final_snv_2["simplified_consequence"] = final_snv_2["consequence"].apply(get_simplified_consequence)

In [251]:
# 1. Replace aa_alt == '=' with aa_ref
final_snv_2.loc[final_snv_2['aa_alt'] == '=', 'aa_alt'] = final_snv_2['aa_ref']

# 2. Replace 'Ter' with '*' in protein HGVS
final_snv_2['hgvs_p'] = final_snv_2['hgvs_p'].str.replace('Ter', '*', regex=False)

# 3. If aa_ref == aa_alt, convert ending of hgvs_p to '='
final_snv_2['hgvs_p'] = np.where(
    final_snv_2['aa_ref'] == final_snv_2['aa_alt'],
    final_snv_2['hgvs_p'].str.replace(
        r'([A-Za-z]{3}\d{1,5})[A-Za-z]{3}$',  # e.g. Arg123Arg → Arg123=
        r'\1=',
        regex=True
    ),
    final_snv_2['hgvs_p']
)

In [252]:
final_snv_2.drop(columns = ['spdi_key', '#Uploaded_variation', 'Location', 'Allele', 'Consequence', 'IMPACT', 'SYMBOL', 'Gene_m', 
                   'Feature_type', 'Feature', 'BIOTYPE', 'EXON', 'INTRON', 'HGVSc', 'HGVSp', 'cDNA_position', 'CDS_position', 
                   'Protein_position', 'Amino_acids', 'Codons', 'Existing_variation', 'REF_ALLELE', 'UPLOADED_ALLELE', 'DISTANCE',
                   'STRAND_m', 'FLAGS', 'SYMBOL_SOURCE', 'HGNC_ID', 'MANE', 'MANE_SELECT', 'MANE_PLUS_CLINICAL', 'TSL', 'APPRIS', 
                   'SIFT', 'PolyPhen', 'HGVS_OFFSET', 'AF', 'CLIN_SIG', 'SOMATIC', 'PHENO', 'PUBMED', 'MOTIF_NAME', 'MOTIF_POS', 
                   'HIGH_INF_POS', 'MOTIF_SCORE_CHANGE', 'TRANSCRIPTION_FACTORS', 'hgvs_c_m', 'transcript_pos_m', 'hgvs_p_m', 'aa_pos_m',
                   'aa_ref_m', 'aa_alt_m', 'consequence_m'], inplace = True)


new_column_order = ['ID','Dataset', 'Gene', 'HGNC_id', 'Chrom','STRAND','hg19_pos', 'hg38_start','hg38_end',
       'ref_allele', 'alt_allele', 'auth_transcript_id', 'transcript_pos',
       'transcript_ref', 'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt',
       'hgvs_c', 'hgvs_p', 'consequence','simplified_consequence', 'auth_reported_score',
       'auth_reported_rep_score', 'auth_reported_func_class',
       'splice_measure', 'gnomad_MAF', 'clinvar_sig', 'clinvar_star',
       'clinvar_date_last_reviewed', 'nucleotide_or_aa', 
       'Ensembl Transcript ID', 'RefSeq Transcript ID', 'Interval 1 Name', 'Interval 1 Range',
                                           'Interval 1 Class', 'Interval 2 Name', 'Interval 2 Range',
                                           'Interval 2 Class', 'Interval 3 Name', 'Interval 3 Range',
                                           'Interval 3 Class', 'Interval 4 Name', 'Interval 4 Range',
                                           'Interval 4 Class', 'Interval 5 Name', 'Interval 5 Range',
                                           'Interval 5 Class', 'Interval 6 Name', 'Interval 6 Range','Interval 6 Class','Flag']

final_snv_2 = final_snv_2[new_column_order]


In [253]:
import pandas as pd 

def qc_count_missing(df):
    # Columns we care about
    cols_to_check = [
        'Gene', 'HGNC_id', 'Chrom',
        'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele',
        'transcript_pos', 'transcript_ref', 'transcript_alt',
        'aa_pos', 'aa_ref', 'aa_alt',
        'hgvs_c', 'hgvs_p', 'consequence'
    ]
    
    # Build a summary dict: counts of missing values
    missing_counts = {
        col: df[col].isna().sum()
        for col in cols_to_check
        if col in df.columns
    }
    
    report = pd.DataFrame.from_dict(missing_counts, orient='index', columns=['Missing_Count'])
    report = report[report['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)
    
    print("\n[QC REPORT] Count of missing values per column:")
    print(report)
    
    return report

qc_count_missing(final_snv_2)


[QC REPORT] Count of missing values per column:
                Missing_Count
aa_alt                  20983
aa_ref                  19418
aa_pos                  19328
hgvs_p                  18701
hgvs_c                   5337
transcript_pos           4065
consequence               823
transcript_alt            136
alt_allele                136
ref_allele                124
transcript_ref            124
hg38_end                    7
hg38_start                  7


,Missing_Count
aa_alt,20983
aa_ref,19418
aa_pos,19328
hgvs_p,18701
hgvs_c,5337
transcript_pos,4065
consequence,823
transcript_alt,136
alt_allele,136
ref_allele,124


In [254]:
# --- Validation Block: Merge Integrity Check ---
pd_1 = pd.read_csv(TMP_DIR/"supp_data_combined_curation.csv.gz", header=0)
# Check unique IDs
ids_before = set(pd_1['ID'].dropna().unique())
ids_after = set(final_snv_2['ID'].dropna().unique())

if ids_before != ids_after:
    missing_ids = ids_before - ids_after
    new_ids = ids_after - ids_before
    raise ValueError(
        f"[ERROR] Merge altered unique IDs.\n"
        f"Missing IDs after merge: {missing_ids}\n"
        f"New unexpected IDs after merge: {new_ids}\n"
        f"Count before: {len(ids_before)}, after: {len(ids_after)}"
    )
else:
    print(f"[OK] Unique IDs preserved across merge (n={len(ids_before)}).")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2847911957.py:2: DtypeWarning: Columns (3,7,8,9,10,11,12,13,16,18,19,20,21,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_1 = pd.read_csv(TMP_DIR/"supp_data_combined_curation.csv.gz", header=0)


[OK] Unique IDs preserved across merge (n=434166).


In [255]:
final_snv_2.to_csv(TMP_DIR/"mapped_data_v3.csv.gz", compression = 'gzip', index = False)

In [256]:
import pandas as pd

final_snv_4 = pd.read_csv(TMP_DIR/"mapped_data_v3.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2067065503.py:3: DtypeWarning: Columns (4,11,15,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51) have mixed types. Specify dtype option on import or set low_memory=False.
  final_snv_4 = pd.read_csv(TMP_DIR/"mapped_data_v3.csv.gz")


In [257]:
import pandas as pd

clinvar = pd.read_csv(DATA_DIR/"ClinVar_files/variant_summary_2025-01.txt.gz",
                      sep='\t', compression='gzip')

clinvar38 = clinvar[clinvar['Assembly'] == 'GRCh38']

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/3391692277.py:3: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  clinvar = pd.read_csv(DATA_DIR/"ClinVar_files/variant_summary_2025-01.txt.gz",


In [258]:
# Split off left and right of colon
clinvar38[['transcript_gene', 'rest']] = clinvar38['Name'].str.split(':', n=1, expand=True)

# Split transcript_gene into transcript and gene
clinvar38['transcript'] = clinvar38['transcript_gene'].str.split('(', n=1).str[0]
clinvar38['gene'] = clinvar38['transcript_gene'].str.split('(', n=1).str[1].str.rstrip(')')

# Split rest into hgvs_c and hgvs_p
clinvar38['hgvs_c_clinvar'] = clinvar38['rest'].str.split(' ', n=1).str[0]
clinvar38['hgvs_p_clinvar'] = clinvar38['rest'].str.split(' ', n=1).str[1].str.strip('()')

# Drop intermediate columns
clinvar38 = clinvar38.drop(columns=['transcript_gene', 'rest'])

clinvar38['transcript_base'] = clinvar38['transcript'].str.replace(r'\.\d+$', '', regex=True)

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/3704753216.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clinvar38[['transcript_gene', 'rest']] = clinvar38['Name'].str.split(':', n=1, expand=True)
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/3704753216.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clinvar38[['transcript_gene', 'rest']] = clinvar38['Name'].str.split(':', n=1, expand=True)
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/3704753216.py:5: SettingWithCopyWarning: 
A value is tryin

In [259]:
aa_dict = {'A': 'Ala','R': 'Arg','N': 'Asn','D': 'Asp','C': 'Cys','E': 'Glu','Q': 'Gln','G': 'Gly','H': 'His',
           'I': 'Ile','L': 'Leu','K': 'Lys','M': 'Met','F': 'Phe','P': 'Pro','S': 'Ser','T': 'Thr','W': 'Trp',
           'Y': 'Tyr','V': 'Val', '=': '=','*': '*', "~": 'del', '-':'del','STOP': '*','Del':'del', "X": 'del'}

reversed_aa_dict = {v: k for k, v in aa_dict.items()}

clinvar38['aa_ref'] = (
    clinvar38['hgvs_p_clinvar']
      .astype('string')
      .str.extract(r'^p\.([A-Z][a-z]{2}|\*|Ter)')[0]
      .replace({'Ter': '*'})
      .map(reversed_aa_dict)
)

In [260]:
clinvar38['Chromosome'] = clinvar38['Chromosome'].astype(str)
clinvar38['aa_ref'] = clinvar38['aa_ref'].astype(str)
final_snv_4['Chrom'] = final_snv_4['Chrom'].astype(str)
final_snv_4['aa_ref'] = final_snv_4['aa_ref'].astype(str)
final_snv_4['ref_allele'] = final_snv_4['ref_allele'].astype(str)
final_snv_4['alt_allele'] = final_snv_4['alt_allele'].astype(str)
clinvar38['ReferenceAlleleVCF'] = clinvar38['ReferenceAlleleVCF'].astype(str)
clinvar38['AlternateAlleleVCF'] = clinvar38['AlternateAlleleVCF'].astype(str)
final_snv_4['transcript_refseq'] = final_snv_4['RefSeq Transcript ID'].str.replace(r'\.\d+$', '', regex=True)

final_snv_4['hg38_start'] = (
    pd.to_numeric(final_snv_4['hg38_start'], errors='coerce')
    .round().astype('Int64')  # round in case of .0 floats
)

clinvar38['PositionVCF'] = (
    pd.to_numeric(clinvar38['PositionVCF'], errors='coerce')
    .astype('Int64')
)

final_snv_2_aa = final_snv_4[final_snv_4['nucleotide_or_aa'] == 'aa']
final_snv_2_nuc = final_snv_4[final_snv_4['nucleotide_or_aa'] == 'nucleotide']

merged_y = pd.merge(
    final_snv_2_aa.fillna({'aa_ref': 'NA_FILL'}),
    clinvar38.fillna({'aa_ref': 'NA_FILL'}),
    left_on=['ref_allele','alt_allele','hg38_start','Chrom','aa_ref'],
    right_on=['ReferenceAlleleVCF','AlternateAlleleVCF','PositionVCF','Chromosome','aa_ref'],
    how='left'
)

merged_x = pd.merge(
    final_snv_2_nuc,
    clinvar38[['ReferenceAlleleVCF','AlternateAlleleVCF','PositionVCF','Chromosome','ClinicalSignificance','ReviewStatus','LastEvaluated']],
    left_on=['ref_allele','alt_allele','hg38_start','Chrom'],
    right_on=['ReferenceAlleleVCF','AlternateAlleleVCF','PositionVCF','Chromosome'],
    how='left'
)
columns_to_update_1 = ['clinvar_sig','clinvar_star','clinvar_date_last_reviewed']
columns_with_fallback_1 = ['ClinicalSignificance','ReviewStatus','LastEvaluated']

for x_col, y_col in zip(columns_to_update_1, columns_with_fallback_1):
    merged_y[x_col] = merged_y[x_col].fillna(merged_y[y_col])

for x_col, y_col in zip(columns_to_update_1, columns_with_fallback_1):
    merged_x[x_col] = merged_x[x_col].fillna(merged_x[y_col])

merged = pd.concat([merged_x, merged_y])

merged.drop(columns=['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID',
       'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)',
       'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList',
       'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession',
       'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele',
       'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines',
       'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID',
       'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF',
       'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated',
       'ReviewStatusClinicalImpact', 'Oncogenicity',
       'OncogenicityLastEvaluated', 'ReviewStatusOncogenicity','transcript_refseq', 'transcript', 
       'gene', 'hgvs_c_clinvar', 'hgvs_p_clinvar', 'transcript_base'], inplace=True)

merged.to_csv(TMP_DIR/"mapped_data_clinvar.csv.gz", compression = 'gzip', index = False)

In [261]:
import pandas as pd

gnomad = pd.read_csv(DATA_DIR/"gnomAD_files/gnomad_v4.csv.gz")

merged_2 = pd.read_csv(TMP_DIR/"mapped_data_clinvar.csv.gz")

merged_2['hg38_start'] = (
    pd.to_numeric(merged_2['hg38_start'], errors='coerce')
    .round().astype('Int64') 
)

gnomad['Position'] = (
    pd.to_numeric(gnomad['Position'], errors='coerce')
    .astype('Int64')
)

gnomad['aa_ref'] = (
    gnomad['Protein Consequence']
      .astype('string')
      .str.extract(r'^p\.([A-Z][a-z]{2}|\*|Ter)')[0]
      .replace({'Ter': '*'})
      .map(reversed_aa_dict)
)

gnomad['Chromosome'] = gnomad['Chromosome'].astype('str')
gnomad['Reference'] = gnomad['Reference'].astype('str')
gnomad['Alternate'] = gnomad['Alternate'].astype('str')
gnomad['Transcript_base'] = gnomad['Transcript'].str.replace(r'\.\d+$', '', regex=True)
gnomad['aa_ref'] = gnomad['aa_ref'].fillna('NA_FILL')
gnomad['aa_ref'] = gnomad['aa_ref'].astype('str')


merged_2['Chrom'] = merged_2['Chrom'].astype('str')
merged_2['ref_allele'] = merged_2['ref_allele'].astype('str')
merged_2['alt_allele'] = merged_2['alt_allele'].astype('str')
merged_2['transcript_ensembl'] = merged_2['Ensembl Transcript ID'].str.replace(r'\.\d+$', '', regex=True)
merged_2['aa_ref'] = merged_2['aa_ref'].astype('str')

merged_2_nuc = merged_2[merged_2['nucleotide_or_aa'] == 'nucleotide']

merged_2_aa = merged_2[merged_2['nucleotide_or_aa'] == 'aa']

jk_x = pd.merge(
    merged_2_nuc,
    gnomad[['Chromosome', 'Position', 'Reference', 'Alternate', 'Allele Frequency','Transcript','Transcript_base']],
    left_on=['Chrom', 'hg38_start', 'ref_allele', 'alt_allele','transcript_ensembl'],
    right_on=['Chromosome', 'Position', 'Reference', 'Alternate','Transcript_base'],
    how='left'
)

jk_y = pd.merge(
    merged_2_aa.fillna({'aa_ref': 'NA_FILL'}),
    gnomad[['Chromosome', 'Position', 'Reference', 'Alternate', 'Allele Frequency','Transcript','Transcript_base','aa_ref']],
    left_on=['Chrom', 'hg38_start', 'ref_allele', 'alt_allele','aa_ref'],
    right_on=['Chromosome', 'Position', 'Reference', 'Alternate','aa_ref'],
    how='left'
)

jk = pd.concat([jk_x, jk_y])

jk['gnomad_MAF'] = jk['Allele Frequency']

# Drop extra columns from gnomad that came with the merge
columns_to_drop = ['Chromosome', 'Position', 'Reference', 'Alternate', 'Allele Frequency','Transcript','Transcript_base','transcript_ensembl']
merged_2_cleaned = jk.drop(columns=columns_to_drop)

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/665522689.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  gnomad = pd.read_csv(DATA_DIR/"gnomAD_files/gnomad_v4.csv.gz")
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/665522689.py:5: DtypeWarning: Columns (4,11,15,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_2 = pd.read_csv(TMP_DIR/"mapped_data_clinvar.csv.gz")


In [262]:
merged_2_cleaned.to_csv(TMP_DIR/"mapped_data_clinvar_gnomad.csv.gz", compression = 'gzip', index = False)

In [263]:
import pandas as pd
merged_2_cleaned = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2980974806.py:2: DtypeWarning: Columns (4,11,15,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_2_cleaned = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad.csv.gz")


In [264]:
import pandas as pd

revel = pd.read_csv(DATA_DIR/"Predictor_score_files/REVEL_scores.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2363743644.py:3: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  revel = pd.read_csv(DATA_DIR/"Predictor_score_files/REVEL_scores.csv.gz")


In [265]:
revel['chr'] = revel['chr'].astype('str')
revel['grch38_pos'] = (
    pd.to_numeric(revel['grch38_pos'], errors='coerce')
    .astype('Int64')
)
revel['ref'] = revel['ref'].astype('str')
revel['alt'] = revel['alt'].astype('str')
revel['aaref'] = revel['aaref'].astype('str')
revel['aaalt'] = revel['aaalt'].astype('str')
revel['Ensembl_transcriptid'] = revel['Ensembl_transcriptid'].astype('str')

In [266]:
merged_2_cleaned['Chrom'] = merged_2_cleaned['Chrom'].astype('str')
merged_2_cleaned['hg38_start'] = (
    pd.to_numeric(merged_2_cleaned['hg38_start'], errors='coerce')
    .round().astype('Int64') 
)
merged_2_cleaned['ref_allele'] = merged_2_cleaned['ref_allele'].astype('str')
merged_2_cleaned['alt_allele'] = merged_2_cleaned['alt_allele'].astype('str')
merged_2_cleaned['aa_ref'] = merged_2_cleaned['aa_ref'].astype('str')
merged_2_cleaned['aa_alt'] = merged_2_cleaned['aa_alt'].astype('str')

In [267]:
revel_df = pd.merge(merged_2_cleaned,revel[['REVEL','chr', 'grch38_pos', 'ref', 'alt','aaref','aaalt']], 
                    left_on = ['Chrom','hg38_start','ref_allele', 'alt_allele','aa_ref','aa_alt'], 
                    right_on = ['chr', 'grch38_pos', 'ref', 'alt','aaref','aaalt'],how = 'left')

In [268]:
revel_df = revel_df.drop(columns=['chr', 'grch38_pos', 'ref', 'alt','aaref','aaalt'])

In [269]:
revel_df.to_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL.csv.gz", compression = 'gzip', index = False)

In [270]:
import pandas as pd
revel_df = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2141448464.py:2: DtypeWarning: Columns (4,11,15,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51) have mixed types. Specify dtype option on import or set low_memory=False.
  revel_df = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL.csv.gz")


In [271]:
import pandas as pd
AM = pd.read_csv(DATA_DIR/"Predictor_score_files/AlphaMissense_hg38.tsv.gz", sep='\t', header=3, compression='gzip')

In [272]:
AM['#CHROM'] = AM['#CHROM'].astype(str).str.replace('chr', '', regex=False)
AM['POS'] = (
    pd.to_numeric(AM['POS'], errors='coerce')
    .astype('Int64')
)
AM['REF'] = AM['REF'].astype('str')
AM['ALT'] = AM['ALT'].astype('str')
AM['protein_variant'] = AM['protein_variant'].astype('str')
AM['transcript_ensembl'] = AM['transcript_id'].str.replace(r'\.\d+$', '', regex=True)

revel_df['Chrom'] = revel_df['Chrom'].astype('str')
revel_df['hg38_start'] = (
    pd.to_numeric(revel_df['hg38_start'], errors='coerce')
    .round().astype('Int64') 
)
revel_df['ref_allele'] = revel_df['ref_allele'].astype('str')
revel_df['alt_allele'] = revel_df['alt_allele'].astype('str')
revel_df['aa_ref'] = revel_df['aa_ref'].astype('str')
revel_df['aa_alt'] = revel_df['aa_alt'].astype('str')
revel_df['aa_pos'] = (
    pd.to_numeric(revel_df['aa_pos'], errors='coerce')
    .astype('Int64')
)
revel_df['three_letter'] = (
    revel_df['aa_ref'].astype(str) +
    revel_df['aa_pos'].astype(str) +
    revel_df['aa_alt'].astype(str)
)


In [273]:
AM_df = pd.merge(revel_df, AM, left_on = ['Chrom','hg38_start','ref_allele', 'alt_allele','three_letter'],
                 right_on = ['#CHROM', 'POS', 'REF','ALT','protein_variant'], how = 'left')

In [274]:
AM_df = AM_df.drop(columns=['three_letter', '#CHROM', 'POS', 'REF', 'ALT', 'genome', 'uniprot_id', 
                            'transcript_id', 'protein_variant','transcript_ensembl'])


In [275]:
AM_df.columns = [col.replace('_x', '') for col in AM_df.columns]

AM_df = AM_df.rename(columns={
    'am_pathogenicity': 'AM_score',
    'am_class': 'AM_class'
})


In [276]:
AM_df.to_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM.csv.gz", compression = 'gzip', index = False)

In [277]:
import pandas as pd
AM_df = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2146943670.py:2: DtypeWarning: Columns (4,11,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51) have mixed types. Specify dtype option on import or set low_memory=False.
  AM_df = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM.csv.gz")


In [278]:
import pandas as pd
import numpy as np
from pyliftover import LiftOver

lo_19 = LiftOver(str(DATA_DIR/"Liftover_files/hg38ToHg19.over.chain.gz"))

# Define a function to lift over positions
def lift_over_pos_38_to_19(row):
    chrom = f"chr{row['Chrom']}"
    pos = row['hg38_start']

    if pd.isna(chrom) or pd.isna(pos):
        return np.nan
    else:
        result = lo_19.convert_coordinate(chrom, pos)
        if result:
            return result[0][1] 
        else:
            return np.nan 

AM_df['hg19_pos'] = AM_df.apply(
    lift_over_pos_38_to_19,
    axis=1
)

In [279]:
import pandas as pd

spliceAI = pd.read_csv(
    DATA_DIR/"Predictor_score_files/exome_spliceai_scores.vcf.gz",
    comment="#",
    header=None,
    usecols=[0,1,2,3,4,5,6,7],
    sep="\t",
    dtype=str
)

In [280]:
AM_df['hg19_pos'] = (
    pd.to_numeric(AM_df['hg19_pos'], errors='coerce')
    .round().astype('Int64') 
)

AM_df['Chrom'] = AM_df['Chrom'].astype('str')
AM_df['ref_allele'] = AM_df['ref_allele'].astype('str')
AM_df['alt_allele'] = AM_df['alt_allele'].astype('str')

In [281]:
spliceAI[1] = (
    pd.to_numeric(spliceAI[1], errors='coerce')
    .round().astype('Int64') 
)

In [282]:
splice_df = pd.merge(AM_df, spliceAI, left_on = ['Chrom', 'hg19_pos', 'ref_allele', 'alt_allele'], 
                  right_on = [0,1,3,4], how = 'left')

In [283]:
splice_df.to_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM_spliceAI.csv.gz", compression = 'gzip', index = False)

In [284]:
splice_df = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM_spliceAI.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1300873374.py:1: DtypeWarning: Columns (4,11,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51,55) have mixed types. Specify dtype option on import or set low_memory=False.
  splice_df = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM_spliceAI.csv.gz")


In [285]:
splice_df_expanded = (
    splice_df['7']
    .dropna() 
    .str.split(';')
    .apply(lambda x: dict(item.split('=') for item in x if '=' in item))
    .apply(pd.Series)
)

In [286]:
#filter out genes not in pillar project 

splice_df_expanded = splice_df_expanded[~splice_df_expanded['SYMBOL'].isin(['CBSL','CH507-396I9.6','RP5-972B16.2','RAD51L3-RFFL'])]

In [287]:
splice_df = splice_df.join(splice_df_expanded, rsuffix="_exp")

In [288]:
splice_df = splice_df[~((splice_df['0'].notna()) & (splice_df['SYMBOL'].isna()))]

In [289]:
splice_df = splice_df.drop(columns = ['0','1','2','3','4','5','6','7','SYMBOL','STRAND_exp','TYPE','DIST'])

In [290]:
cols_to_prefix = ["DS_AG", "DS_AL", "DS_DG", "DS_DL", 
                  "DP_AG", "DP_AL", "DP_DG", "DP_DL"]

splice_df = splice_df.rename(
    columns={col: f"spliceAI_{col}" for col in cols_to_prefix}
)

In [291]:
#add final flags where there are transcript mistmatches from the author  

p53 = [
    "TP53_Giacomelli_2018_combined_score",
    "TP53_Giacomelli_2018_p53WT_Nutlin3",
    "TP53_Giacomelli_2018_p53null_Nutlin3",
    "TP53_Giacomelli_2018_p53null_etoposide",
    "TP53_Boettcher_2019",
    "TP53_Fayer_2021_meta"
]

# update only rows that match conditions
mask_p53 = splice_df["Dataset"].isin(p53) & (splice_df["aa_pos"] == 72.0)
splice_df.loc[mask_p53, "Flag"] = "*"

In [292]:
kcne1 = [
    "KCNE1_Muhammad_2024_trafficking",
    "KCNE1_Muhammad_2024_potassium_flux",
    "KCNE1_Muhammad_2024_trafficking_WT_background_DN"
]

# update only rows that match conditions
mask_kcne1 = splice_df["Dataset"].isin(kcne1) & (splice_df["aa_pos"] == 38.0)
splice_df.loc[mask_kcne1, "Flag"] = "*"

In [293]:
brca1 = [
    "BRCA1_Adamovich_2022_Cisplatin_Resistance",
    "BRCA1_Adamovich_2022_HDR"
]

# update only rows that match conditions
mask_brca1 = splice_df["Dataset"].isin(brca1) & (splice_df["aa_pos"] == 1613.0)
splice_df.loc[mask_brca1, "Flag"] = "*"

In [294]:
splice_df.to_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM_spliceAI.csv.gz", compression = 'gzip', index = False)

In [295]:
splice_df = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM_spliceAI.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1300873374.py:1: DtypeWarning: Columns (4,11,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51) have mixed types. Specify dtype option on import or set low_memory=False.
  splice_df = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM_spliceAI.csv.gz")


In [296]:
import pandas as pd
mutpred = pd.read_csv(DATA_DIR/"Predictor_score_files/MutPred2_scores_1.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2155070957.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  mutpred = pd.read_csv(DATA_DIR/"Predictor_score_files/MutPred2_scores_1.csv.gz")


In [297]:
mutpred = mutpred.drop_duplicates(subset = ['Gene','aa_pos','aa_ref','aa_alt'])

In [298]:
mutpred2 = pd.merge(splice_df, mutpred[['Gene', 'aa_pos','aa_ref','aa_alt', 'MutPred2','MP2_train','REVEL_train']],
                    on = ['Gene', 'aa_pos','aa_ref','aa_alt'], how = 'left')

In [299]:
#need to add in CALM1_CALM2_CALM3 MP2 scores and XRCC2 MP2 scores 

import pandas as pd

mp2_train = pd.read_csv(DATA_DIR/"Predictor_score_files/MutPred2_scores_2.csv.gz")


In [300]:
CALM1_XR = mp2_train[(mp2_train['Gene'] == 'CALM1') | (mp2_train['Gene'] == 'XRCC2')].copy()
CALM1_XR['Gene'] = CALM1_XR['Gene'].replace('CALM1', 'CALM1_CALM2_CALM3')

In [301]:
mutpred2['aa_pos'] = pd.to_numeric(mutpred2['aa_pos'], errors='coerce').astype('Int64')
mutpred2['one_letter'] = mutpred2['aa_ref'] + mutpred2['aa_pos'].astype(str) + mutpred2['aa_alt']

mutpred2 = pd.merge(mutpred2, CALM1_XR [['Gene', 'variant','MP2','is_mp2_train']], left_on = ['Gene','one_letter'],
                   right_on = ['Gene', 'variant'], how = 'left')

columns_to_update_mp2 = ['MutPred2','MP2_train']
columns_with_fallback_mp2 = ['MP2','is_mp2_train']

for x_col, y_col in zip(columns_to_update_mp2, columns_with_fallback_mp2):
    mutpred2[x_col] = mutpred2[x_col].fillna(mutpred2[y_col])


In [302]:
mutpred2 = mutpred2.drop(['one_letter', 'variant', 'MP2', 'is_mp2_train'], axis=1)

In [303]:
mutpred2.to_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM_spliceAI_MP2.csv.gz", compression = 'gzip', index = False)

In [304]:
import pandas as pd

mutpred2 = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM_spliceAI_MP2.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/4195445306.py:3: DtypeWarning: Columns (4,11,12,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51,54,64,65) have mixed types. Specify dtype option on import or set low_memory=False.
  mutpred2 = pd.read_csv(TMP_DIR/"mapped_data_clinvar_gnomad_REVEL_AM_spliceAI_MP2.csv.gz")


In [305]:
import pandas as pd

clinvar_18 = pd.read_csv(DATA_DIR/"ClinVar_files/variant_summary_2018-12.txt.gz",
                      sep='\t', compression='gzip')

clinvar38_18 = clinvar_18[clinvar_18['Assembly'] == 'GRCh38']

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/35927306.py:3: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  clinvar_18 = pd.read_csv(DATA_DIR/"ClinVar_files/variant_summary_2018-12.txt.gz",


In [306]:
# Split off left and right of colon
clinvar38_18[['transcript_gene', 'rest']] = clinvar38_18['Name'].str.split(':', n=1, expand=True)

# Split transcript_gene into transcript and gene
clinvar38_18['transcript'] = clinvar38_18['transcript_gene'].str.split('(', n=1).str[0]
clinvar38_18['gene'] = clinvar38_18['transcript_gene'].str.split('(', n=1).str[1].str.rstrip(')')

# Split rest into hgvs_c and hgvs_p
clinvar38_18['hgvs_c_clinvar'] = clinvar38_18['rest'].str.split(' ', n=1).str[0]
clinvar38_18['hgvs_p_clinvar'] = clinvar38_18['rest'].str.split(' ', n=1).str[1].str.strip('()')

# Drop intermediate columns
clinvar38_18 = clinvar38_18.drop(columns=['transcript_gene', 'rest'])

clinvar38_18['transcript_base'] = clinvar38_18['transcript'].str.replace(r'\.\d+$', '', regex=True)

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1680980700.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clinvar38_18[['transcript_gene', 'rest']] = clinvar38_18['Name'].str.split(':', n=1, expand=True)
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1680980700.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clinvar38_18[['transcript_gene', 'rest']] = clinvar38_18['Name'].str.split(':', n=1, expand=True)
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1680980700.py:5: SettingWithCopyWarning: 
A va

In [307]:
aa_dict = {'A': 'Ala','R': 'Arg','N': 'Asn','D': 'Asp','C': 'Cys','E': 'Glu','Q': 'Gln','G': 'Gly','H': 'His',
           'I': 'Ile','L': 'Leu','K': 'Lys','M': 'Met','F': 'Phe','P': 'Pro','S': 'Ser','T': 'Thr','W': 'Trp',
           'Y': 'Tyr','V': 'Val', '=': '=','*': '*', "~": 'del', '-':'del','STOP': '*','Del':'del', "X": 'del'}

reversed_aa_dict = {v: k for k, v in aa_dict.items()}

clinvar38_18['aa_ref'] = (
    clinvar38_18['hgvs_p_clinvar']
      .astype('string')
      .str.extract(r'^p\.([A-Z][a-z]{2}|\*|Ter)')[0]
      .replace({'Ter': '*'})
      .map(reversed_aa_dict)
)

In [308]:
import numpy as np
mutpred2[['clinvar_sig_2018','clinvar_star_2018','clinvar_date_last_reviewed_2018']] = np.nan

In [309]:
clinvar38_18['Chromosome'] = clinvar38_18['Chromosome'].astype(str)
clinvar38_18['aa_ref'] = clinvar38_18['aa_ref'].astype(str)
mutpred2['Chrom'] = mutpred2['Chrom'].astype(str)
mutpred2['aa_ref'] = mutpred2['aa_ref'].astype(str)
mutpred2['ref_allele'] = mutpred2['ref_allele'].astype(str)
mutpred2['alt_allele'] = mutpred2['alt_allele'].astype(str)
clinvar38_18['ReferenceAllele'] = clinvar38_18['ReferenceAllele'].astype(str)
clinvar38_18['AlternateAllele'] = clinvar38_18['AlternateAllele'].astype(str)
mutpred2['transcript_refseq'] = mutpred2['RefSeq Transcript ID'].str.replace(r'\.\d+$', '', regex=True)

mutpred2['hg38_start'] = (
    pd.to_numeric(mutpred2['hg38_start'], errors='coerce')
    .round().astype('Int64')  
)

clinvar38_18['Start'] = (
    pd.to_numeric(clinvar38_18['Start'], errors='coerce')
    .astype('Int64')
)

mutpred2_aa = mutpred2[mutpred2['nucleotide_or_aa'] == 'aa']
mutpred2_nuc = mutpred2[mutpred2['nucleotide_or_aa'] == 'nucleotide']

merged_y2 = pd.merge(
    mutpred2_aa.fillna({'aa_ref': 'NA_FILL'}),
    clinvar38_18.fillna({'aa_ref': 'NA_FILL'}),
    left_on=['ref_allele','alt_allele','hg38_start','Chrom','aa_ref'],
    right_on=['ReferenceAllele','AlternateAllele','Start','Chromosome','aa_ref'],
    how='left'
)

merged_x2 = pd.merge(
    mutpred2_nuc,
    clinvar38_18[['ReferenceAllele','AlternateAllele','Start','Chromosome','ClinicalSignificance','ReviewStatus','LastEvaluated']],
    left_on=['ref_allele','alt_allele','hg38_start','Chrom'],
    right_on=['ReferenceAllele','AlternateAllele','Start','Chromosome'],
    how='left'
)
columns_to_update_1 = ['clinvar_sig_2018','clinvar_star_2018','clinvar_date_last_reviewed_2018']
columns_with_fallback_1 = ['ClinicalSignificance','ReviewStatus','LastEvaluated']

for x_col, y_col in zip(columns_to_update_1, columns_with_fallback_1):
    merged_y2[x_col] = merged_y2[x_col].fillna(merged_y2[y_col])

for x_col, y_col in zip(columns_to_update_1, columns_with_fallback_1):
    merged_x2[x_col] = merged_x2[x_col].fillna(merged_x2[y_col])

merged2 = pd.concat([merged_x2, merged_y2])

merged2.drop(columns=['transcript_refseq', 'ReferenceAllele', 'AlternateAllele', 'Start', 'Chromosome', 'ClinicalSignificance', 
'ReviewStatus', 'LastEvaluated', '#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinSigSimple', 'RS# (dbSNP)', 
'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList', 'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession', 'Stop', 
'Cytogenetic', 'NumberSubmitters', 'Guidelines', 'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID', 'transcript', 
'gene', 'hgvs_c_clinvar', 'hgvs_p_clinvar', 'transcript_base'], inplace=True)


In [310]:
merged2 = merged2.rename(columns={
    'clinvar_sig': 'clinvar_sig_2025',
    'clinvar_star': 'clinvar_star_2025',
    'clinvar_date_last_reviewed': 'clinvar_date_last_reviewed_2025'
})

In [311]:
gene_change = merged2['Dataset'].isin(['BRCA2_Hu_2024', 'JAG1_Gilbert_2024', 'RHO_Wan_2019','LARGE1_Ma_2024','FKRP_Ma_2024'])

merged2.loc[gene_change, 'nucleotide_or_aa'] = 'aa'
merged2.loc[gene_change, 'splice_measure'] = 'No'

In [312]:
merged2.to_csv(TMP_DIR/"mapped_data_clinvar18_25_gnomad_REVEL_AM_spliceAI_MP2.csv.gz", compression = 'gzip', index = False)

In [313]:
merged2 = pd.read_csv(TMP_DIR/"mapped_data_clinvar18_25_gnomad_REVEL_AM_spliceAI_MP2.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/2275589290.py:1: DtypeWarning: Columns (4,11,12,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51,54,64,65,66,67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  merged2 = pd.read_csv(TMP_DIR/"mapped_data_clinvar18_25_gnomad_REVEL_AM_spliceAI_MP2.csv.gz")


In [314]:
#check if the dataset lengths match 

import pandas as pd

pp = pd.read_csv(TMP_DIR/"supp_data_combined_curation.csv.gz")

dff =  pd.read_csv(TMP_DIR/"mapped_data_clinvar18_25_gnomad_REVEL_AM_spliceAI_MP2.csv.gz")

df1_lengths = pp.groupby('Dataset').size()
df2_lengths = dff.groupby('Dataset')['ID'].nunique()

comparison = pd.DataFrame({
    'df1': df1_lengths,
    'df2': df2_lengths
}).fillna(0)

same_length = comparison[comparison['df1'] == comparison['df2']]
not_same_length = comparison[comparison['df1'] != comparison['df2']]

print("Groups with the same length:")
print(same_length)

print("\nGroups not the same length:")
print(not_same_length)

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/489835778.py:5: DtypeWarning: Columns (3,7,8,9,10,11,12,13,16,18,19,20,21,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  pp = pd.read_csv(TMP_DIR/"supp_data_combined_curation.csv.gz")
/tmp/7554300.1.fowler-login.q/ipykernel_1384509/489835778.py:7: DtypeWarning: Columns (4,11,12,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51,54,64,65,66,67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  dff =  pd.read_csv(TMP_DIR/"mapped_data_clinvar18_25_gnomad_REVEL_AM_spliceAI_MP2.csv.gz")


Groups with the same length:
                                             df1    df2
Dataset                                                
ASPA_Grønbæk-Thygesen_2024_abundance        6151   6151
ASPA_Grønbæk-Thygesen_2024_toxicity         6148   6148
BAP1_Waters_2024                           14165  14165
BARD1_IGVF                                  8851   8851
BRCA1_Adamovich_2022_Cisplatin_Resistance   1427   1427
...                                          ...    ...
TP53_Kato_2003_h1433snWT                    2314   2314
TPK1_Weile_2017                             4584   4584
TSC2_IGVF                                   9253   9253
VHL_Buckley_2024                            2268   2268
XRCC2_IGVF                                  3743   3743

[83 rows x 2 columns]

Groups not the same length:
Empty DataFrame
Columns: [df1, df2]
Index: []


In [315]:
import pandas as pd
import os

def summary_tsv(file_name):
    
    # Read the DataFrame as a CSV
    df = pd.read_csv(file_name)
    
    # Output file path
    output_file = os.path.expanduser(OUT_DIR/"dataset_summary.tsv")

    # List to store summary data
    summary_data = []

    grouped = df.groupby('Dataset')
    
    for dataset_name, group in grouped:
        
        # Deduplicate on ID
        group_dedup = group.drop_duplicates(subset='ID')

        # Total number of variants
        total_variants = group_dedup['ID'].nunique()

        # ---- AA-based counts ----
        syn_count = (
            group_dedup['simplified_consequence'] == 'synonymous_variant').sum()

        stop_count = (
            group_dedup['simplified_consequence'] == 'stop_gained').sum()

        # Append data to list
        summary_data.append({
            "Dataset": dataset_name,
            "Total_Variants": total_variants,
            "Synonymous": syn_count,
            "Stop": stop_count
        })

    # Convert to DataFrame and save as TSV
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(output_file, sep='\t', index=False)

    print(f"Summary written to {output_file}")

summary_tsv(TMP_DIR/"mapped_data_clinvar18_25_gnomad_REVEL_AM_spliceAI_MP2.csv.gz")

/tmp/7554300.1.fowler-login.q/ipykernel_1384509/1788158896.py:7: DtypeWarning: Columns (4,11,12,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51,54,64,65,66,67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name)


Summary written to /net/bbi/vol1/home/mtejura/IGVF-cvfg-pillar-project/outputs/dataset_summary.tsv


In [27]:
import pandas as pd
gold_standard = pd.read_csv(TMP_DIR/"mapped_data_clinvar18_25_gnomad_REVEL_AM_spliceAI_MP2.csv.gz")

/tmp/7557874.1.fowler-login.q/ipykernel_1531332/1512625954.py:2: DtypeWarning: Columns (4,11,12,22,23,24,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51,54,64,65,66,67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  gold_standard = pd.read_csv(TMP_DIR/"mapped_data_clinvar18_25_gnomad_REVEL_AM_spliceAI_MP2.csv.gz")


In [28]:
acmg_evidence_codes = {
    # Very strong evidence of pathogenicity
    "PVS1": "Pathogenic_Very_Strong",
    "PVS1_Supporting": "Pathogenic_Supporting",
    "PVS1_Moderate": "Pathogenic_Moderate",
    "PVS1_Strong": "Pathogenic_Strong",
    "PVS1_Very": "Pathogenic_Very_Strong",

    # Strong evidence of pathogenicity
    "PS1": "Pathogenic_Strong",
    "PS1_Supporting": "Pathogenic_Supporting",
    "PS1_Moderate": "Pathogenic_Moderate",
    "PS1_Very": "Pathogenic_Very_Strong",

    "PS2": "Pathogenic_Strong",
    "PS2_Supporting": "Pathogenic_Supporting",
    "PS2_Moderate": "Pathogenic_Moderate",
    "PS2_Very": "Pathogenic_Very_Strong",

    "PS3": "Pathogenic_Strong",
    "PS3_Supporting": "Pathogenic_Supporting",
    "PS3_Moderate": "Pathogenic_Moderate",
    "PS3_Very": "Pathogenic_Very_Strong",

    "PS4": "Pathogenic_Strong",
    "PS4_Supporting": "Pathogenic_Supporting",
    "PS4_Moderate": "Pathogenic_Moderate",
    "PS4_Very": "Pathogenic_Very_Strong",

    # Moderate evidence of pathogenicity
    "PM1": "Pathogenic_Moderate",
    "PM1_Supporting": "Pathogenic_Supporting",
    "PM1_Strong": "Pathogenic_Strong",
    "PM1_Very": "Pathogenic_Very_Strong",

    "PM2": "Pathogenic_Moderate",
    "PM2_Supporting": "Pathogenic_Supporting",
    "PM2_Strong": "Pathogenic_Strong",
    "PM2_Very": "Pathogenic_Very_Strong",

    "PM3": "Pathogenic_Moderate",
    "PM3_Supporting": "Pathogenic_Supporting",
    "PM3_Strong": "Pathogenic_Strong",
    "PM3_Very": "Pathogenic_Very_Strong",

    "PM4": "Pathogenic_Moderate",
    "PM4_Supporting": "Pathogenic_Supporting",
    "PM4_Strong": "Pathogenic_Strong",
    "PM4_Very": "Pathogenic_Very_Strong",

    "PM5": "Pathogenic_Moderate",
    "PM5_Supporting": "Pathogenic_Supporting",
    "PM5_Strong": "Pathogenic_Strong",
    "PM5_Very": "Pathogenic_Very_Strong",

    "PM6": "Pathogenic_Moderate",
    "PM6_Supporting": "Pathogenic_Supporting",
    "PM6_Strong": "Pathogenic_Strong",
    "PM6_Very": "Pathogenic_Very_Strong",

    # Supporting evidence of pathogenicity
    "PP1": "Pathogenic_Supporting",
    "PP1_Moderate": "Pathogenic_Moderate",
    "PP1_Strong": "Pathogenic_Strong",
    "PP1_Very": "Pathogenic_Very_Strong",

    "PP2": "Pathogenic_Supporting",
    "PP2_Moderate": "Pathogenic_Moderate",
    "PP2_Strong": "Pathogenic_Strong",
    "PP2_Very": "Pathogenic_Very_Strong",

    "PP3": "Pathogenic_Supporting",
    "PP3_Moderate": "Pathogenic_Moderate",
    "PP3_Strong": "Pathogenic_Strong",
    "PP3_Very": "Pathogenic_Very_Strong",

    "PP4": "Pathogenic_Supporting",
    "PP4_Moderate": "Pathogenic_Moderate",
    "PP4_Strong": "Pathogenic_Strong",
    "PP4_Very": "Pathogenic_Very_Strong",

    "PP5": "Pathogenic_Supporting",
    "PP5_Moderate": "Pathogenic_Moderate",
    "PP5_Strong": "Pathogenic_Strong",
    "PP5_Very": "Pathogenic_Very_Strong",

    # Benign standalone
    "BA1": "Benign_Standalone",

    # Strong evidence of benignity
    "BS1": "Benign_Strong",
    "BS1_Supporting": "Benign_Supporting",
    "BS1_Moderate": "Benign_Moderate",
    "BS1_Very": "Benign_Very_Strong",

    "BS2": "Benign_Strong",
    "BS2_Supporting": "Benign_Supporting",
    "BS2_Moderate": "Benign_Moderate",
    "BS2_Very": "Benign_Very_Strong",

    "BS3": "Benign_Strong",
    "BS3_Supporting": "Benign_Supporting",
    "BS3_Moderate": "Benign_Moderate",
    "BS3_Very": "Benign_Very_Strong",

    "BS4": "Benign_Strong",
    "BS4_Supporting": "Benign_Supporting",
    "BS4_Moderate": "Benign_Moderate",
    "BS4_Very": "Benign_Very_Strong",

    # Supporting evidence of benignity
    "BP1": "Benign_Supporting",
    "BP1_Moderate": "Benign_Moderate",
    "BP1_Strong": "Benign_Strong",
    "BP1_Very": "Benign_Very_Strong",

    "BP2": "Benign_Supporting",
    "BP2_Moderate": "Benign_Moderate",
    "BP2_Strong": "Benign_Strong",
    "BP2_Very": "Benign_Very_Strong",

    "BP3": "Benign_Supporting",
    "BP3_Moderate": "Benign_Moderate",
    "BP3_Strong": "Benign_Strong",
    "BP3_Very": "Benign_Very_Strong",

    "BP4": "Benign_Supporting",
    "BP4_Moderate": "Benign_Moderate",
    "BP4_Strong": "Benign_Strong",
    "BP4_Very": "Benign_Very_Strong",

    "BP5": "Benign_Supporting",
    "BP5_Moderate": "Benign_Moderate",
    "BP5_Strong": "Benign_Strong",
    "BP5_Very": "Benign_Very_Strong",

    "BP6": "Benign_Supporting",
    "BP6_Moderate": "Benign_Moderate",
    "BP6_Strong": "Benign_Strong",
    "BP6_Very": "Benign_Very_Strong",

    "BP7": "Benign_Supporting",
    "BP7_Moderate": "Benign_Moderate",
    "BP7_Strong": "Benign_Strong",
    "BP7_Very": "Benign_Very_Strong"
}

In [29]:
# Define function to classify based on ACMG evidence codes
def classify_acmg(evidence_list):
   
    if not evidence_list:
        return "VUS"  # Default to Uncertain Significance if no evidence is left

    # Categorize evidence into strengths
    pathogenic_very_strong = sum(1 for e in evidence_list if acmg_evidence_codes.get(e) == "Pathogenic_Very_Strong")
    pathogenic_strong = sum(1 for e in evidence_list if acmg_evidence_codes.get(e) == "Pathogenic_Strong")
    pathogenic_moderate = sum(1 for e in evidence_list if acmg_evidence_codes.get(e) == "Pathogenic_Moderate")
    pathogenic_supporting = sum(1 for e in evidence_list if acmg_evidence_codes.get(e) == "Pathogenic_Supporting")

    benign_standalone = sum(1 for e in evidence_list if acmg_evidence_codes.get(e) == 'Benign_standalone')
    benign_strong = sum(1 for e in evidence_list if acmg_evidence_codes.get(e) == "Benign_Strong")
    benign_moderate = sum(1 for e in evidence_list if acmg_evidence_codes.get(e) == "Benign_Moderate")
    benign_supporting = sum(1 for e in evidence_list if acmg_evidence_codes.get(e) == "Benign_Supporting")

    # Pathogenic Classification Rules
    if pathogenic_very_strong >= 1:
        if pathogenic_strong >= 1 or pathogenic_moderate >= 2:
            return "Pathogenic"
    if pathogenic_strong >= 2:
        return "Pathogenic"
    if pathogenic_strong == 1 and pathogenic_moderate >= 2:
        return "Pathogenic"
    if pathogenic_very_strong == 1 and pathogenic_moderate >= 1:
        return "Likely Pathogenic"
    if pathogenic_strong == 1 and pathogenic_moderate == 1:
        return "Likely Pathogenic"
    if pathogenic_moderate >= 3:
        return "Likely Pathogenic"
    if pathogenic_moderate == 2 and pathogenic_supporting >= 2:
        return "Likely Pathogenic"

    # Benign Classification Rules
    if benign_standalone >= 1:
        return 'Benign'
    if benign_strong >= 2:
        return "Benign"
    if benign_strong == 1 and benign_supporting >= 1:
        return "Likely Benign"
    if benign_moderate >= 2:
        return "Likely Benign"

    # Default to VUS if no strong evidence
    return "VUS"


In [30]:
import pandas as pd
ClinGen_repo = pd.read_csv(DATA_DIR/"ClinGen_evidence_repo_files/ClinGen_repo_111025.csv.gz")

In [31]:
ClinGen_repo[['Transcript_ID_clingen', 'HGVS_c_clingen', 'Protein_Change_clingen']] = ClinGen_repo['Variation'].str.extract(
    r'(?P<Transcript_ID>NM_\d+)\.\d+\(.*?\):(?P<HGVS_c>c\.\S+)\s+\((?P<Protein_Change>p\.\S+)\)'
)

In [32]:
remove_codes = {
    # Benign Evidence Codes to Remove
    "BP4", "BP4_Supporting", "BP4_Moderate", "BP4_Strong", "BP4_Very",
    "BS3", "BS3_Supporting", "BS3_Moderate", "BS3_Strong", "BS3_Very",

    # Pathogenic Evidence Codes to Remove
    "PS3", "PS3_Supporting", "PS3_Moderate", "PS3_Strong", "PS3_Very",
    "PP3", "PP3_Supporting", "PP3_Moderate", "PP3_Strong", "PP3_Very",
}

def filter_and_recalculate(evidence_string):
    if pd.isna(evidence_string):
        return "VUS", "" 
    
    # Split the string into a list
    evidence_list = evidence_string.split(",")

    # Remove unwanted codes
    filtered_evidence = [code.strip() for code in evidence_list if code.strip() not in remove_codes]

    # Convert back to comma-separated string
    filtered_evidence_string = ",".join(filtered_evidence) if filtered_evidence else ""

    # If nothing remains after filtering, return VUS
    updated_classification = classify_acmg(filtered_evidence) if filtered_evidence else "VUS"

    return updated_classification, filtered_evidence_string

In [33]:
ClinGen_repo[['Updated_Classification', 'Updated_Evidence Codes']] = ClinGen_repo['Applied Evidence Codes (Met)'].apply(lambda x: pd.Series(filter_and_recalculate(x)))


In [34]:
pd.set_option('display.max_columns', None)

ClinGen_repo = ClinGen_repo.add_suffix("_ClinGen_repo")

In [35]:
#add gold stan transcript
gold_standard['gold_stan_transcript'] = gold_standard['RefSeq Transcript ID'].str.replace(
    r"(NM_\d+)\.\d+", r"\1", regex=True
)

# Convert all relevant columns in both DataFrames to string type before merging
columns_left_clingen = ['Gene',"gold_stan_transcript", 'hgvs_c','hgvs_p']
columns_right_clingen = ['HGNC Gene Symbol_ClinGen_repo',"Transcript_ID_clingen_ClinGen_repo",'HGVS_c_clingen_ClinGen_repo', "Protein_Change_clingen_ClinGen_repo"]

for col in columns_left_clingen:
    gold_standard[col] = gold_standard[col].astype(str)

for col in columns_right_clingen:
    ClinGen_repo[col] = ClinGen_repo[col].astype(str)

#merge
gold_standard_clingen = pd.merge(
    gold_standard,
    ClinGen_repo,
    left_on=columns_left_clingen,
    right_on=columns_right_clingen,
    how='left'
)

In [36]:
#columns to drop 

columns_drop_final = [
'Variation_ClinGen_repo','HGVS Expressions_ClinGen_repo', 'HGNC Gene Symbol_ClinGen_repo',
'Transcript_ID_clingen_ClinGen_repo', 'HGVS_c_clingen_ClinGen_repo', 'Protein_Change_clingen_ClinGen_repo',
'gold_stan_transcript']


gold_standard_clingen_2 = gold_standard_clingen.drop(columns=columns_drop_final)

In [37]:
gold_standard_clingen_2 = gold_standard_clingen_2.rename(
    columns={'STRAND': 'Strand'}
)

In [38]:
gold_standard_clingen_2.loc[
    gold_standard_clingen_2['Gene'] == 'BAP1',
    'Strand'
] = -1

In [39]:
import numpy as np

gold_standard_clingen_2.loc[
    gold_standard_clingen_2['Flag'].isin(['updated', 'updated_2']),
    'Flag'
] = np.nan

In [41]:
new_order = [
    'ID', 'Dataset', 'Gene', 'HGNC_id', 'Chrom', 'Strand', 'hg19_pos',
    'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele',
    'auth_transcript_id', 'transcript_pos', 'transcript_ref',
    'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt', 'hgvs_c', 'hgvs_p',
    'consequence', 'simplified_consequence', 'auth_reported_score',
    'auth_reported_rep_score', 'auth_reported_func_class', 'splice_measure',
    'gnomad_MAF', 'clinvar_sig_2025', 'clinvar_star_2025',
    'clinvar_date_last_reviewed_2025', 'clinvar_sig_2018', 'clinvar_star_2018',
    'clinvar_date_last_reviewed_2018', 'nucleotide_or_aa',
    'Ensembl Transcript ID', 'RefSeq Transcript ID',
    'Interval 1 Name', 'Interval 1 Range', 'Interval 1 Class',
    'Interval 2 Name', 'Interval 2 Range', 'Interval 2 Class',
    'Interval 3 Name', 'Interval 3 Range', 'Interval 3 Class',
    'Interval 4 Name', 'Interval 4 Range', 'Interval 4 Class',
    'Interval 5 Name', 'Interval 5 Range', 'Interval 5 Class',
    'Interval 6 Name', 'Interval 6 Range', 'Interval 6 Class',
    'Flag', 'REVEL', 'REVEL_train', 'AM_score', 'AM_class',
    'MutPred2', 'MP2_train',
    'spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL',
    'spliceAI_DP_AG', 'spliceAI_DP_AL', 'spliceAI_DP_DG', 'spliceAI_DP_DL',
    'ClinVar Variation Id_ClinGen_repo',
    'Allele Registry Id_ClinGen_repo', 'Disease_ClinGen_repo',
    'Mondo Id_ClinGen_repo', 'Mode of Inheritance_ClinGen_repo',
    'Assertion_ClinGen_repo', 'Applied Evidence Codes (Met)_ClinGen_repo',
    'Applied Evidence Codes (Not Met)_ClinGen_repo',
    'Summary of interpretation_ClinGen_repo',
    'PubMed Articles_ClinGen_repo', 'Expert Panel_ClinGen_repo',
    'Guideline_ClinGen_repo', 'Approval Date_ClinGen_repo',
    'Published Date_ClinGen_repo', 'Retracted_ClinGen_repo',
    'Evidence Repo Link_ClinGen_repo', 'Uuid_ClinGen_repo',
    'Updated_Classification_ClinGen_repo',
    'Updated_Evidence Codes_ClinGen_repo'
]

gold_standard_clingen_2 = gold_standard_clingen_2.reindex(columns=new_order)

In [42]:
#change columns to int64 instead of float64

cols_int = [
    "hg19_pos",
    "hg38_start",
    "hg38_end", 
    "aa_pos"
]

gold_standard_clingen_2[cols_int] = gold_standard_clingen_2[cols_int].astype("Int64")


#change STRAND to integer

gold_standard_clingen_2["Strand"] = gold_standard_clingen_2["Strand"].astype("Int64")

#change ClinVar Variation Id_ClinGen_repo to integer

gold_standard_clingen_2["ClinVar Variation Id_ClinGen_repo"] = (
    gold_standard_clingen_2["ClinVar Variation Id_ClinGen_repo"]
    .astype("Int64")
)


#change interval 6 to object class
cols = [
    "Interval 6 Name",
    "Interval 6 Range",
    "Interval 6 Class"
]
gold_standard_clingen_2[cols] = gold_standard_clingen_2[cols].astype("object")


#gzip and save as tsv

gold_standard_clingen_2.to_csv(
    OUT_DIR/"integrated_variant_effect_dataset.tsv.gz",
    sep="\t",
    index=False,
    compression="gzip"
)

In [43]:
#check if the dataset lengths match 

import pandas as pd

pp = pd.read_csv(TMP_DIR/"supp_data_combined_curation.csv.gz")

dff =  pd.read_csv(OUT_DIR/"integrated_variant_effect_dataset.tsv.gz", sep = '\t')

df1_lengths = pp.groupby('Dataset').size()
df2_lengths = dff.groupby('Dataset')['ID'].nunique()

comparison = pd.DataFrame({
    'df1': df1_lengths,
    'df2': df2_lengths
}).fillna(0)

same_length = comparison[comparison['df1'] == comparison['df2']]
not_same_length = comparison[comparison['df1'] != comparison['df2']]

print("Groups with the same length:")
print(same_length)

print("\nGroups not the same length:")
print(not_same_length)

/tmp/7557874.1.fowler-login.q/ipykernel_1531332/459752328.py:5: DtypeWarning: Columns (3,7,8,9,10,11,12,13,16,18,19,20,21,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  pp = pd.read_csv(TMP_DIR/"supp_data_combined_curation.csv.gz")
/tmp/7557874.1.fowler-login.q/ipykernel_1531332/459752328.py:7: DtypeWarning: Columns (4,11,12,22,23,24,30,31,32,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,54,56,58,60,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87) have mixed types. Specify dtype option on import or set low_memory=False.
  dff =  pd.read_csv(OUT_DIR/"integrated_variant_effect_dataset.tsv.gz", sep = '\t')


Groups with the same length:
                                             df1    df2
Dataset                                                
ASPA_Grønbæk-Thygesen_2024_abundance        6151   6151
ASPA_Grønbæk-Thygesen_2024_toxicity         6148   6148
BAP1_Waters_2024                           14165  14165
BARD1_IGVF                                  8851   8851
BRCA1_Adamovich_2022_Cisplatin_Resistance   1427   1427
...                                          ...    ...
TP53_Kato_2003_h1433snWT                    2314   2314
TPK1_Weile_2017                             4584   4584
TSC2_IGVF                                   9253   9253
VHL_Buckley_2024                            2268   2268
XRCC2_IGVF                                  3743   3743

[83 rows x 2 columns]

Groups not the same length:
Empty DataFrame
Columns: [df1, df2]
Index: []


In [4]:
import pandas as pd

dff =  pd.read_csv(OUT_DIR/"integrated_variant_effect_dataset.tsv.gz", sep = '\t')

/tmp/7566765.1.fowler-login.q/ipykernel_1025323/2248193017.py:3: DtypeWarning: Columns (4,11,12,22,23,24,30,31,32,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,54,56,58,60,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87) have mixed types. Specify dtype option on import or set low_memory=False.
  dff =  pd.read_csv(OUT_DIR/"integrated_variant_effect_dataset.tsv.gz", sep = '\t')


In [5]:
dff["Dataset"] = dff["Dataset"].replace(
    "CARD11_Meitlis_2020_SGE_Ibrutinib_DN",
    "CARD11_Meitlis_2020_SGE_Ibrutinib_GoF"
)

In [6]:
dff['Dataset'].unique()

array(['BRCA1_Findlay_2018', 'BRCA2_Hu_2024', 'VHL_Buckley_2024',
       'JAG1_Gilbert_2024', 'BRCA2_Sahu_2023_exon13_SGE',
       'BRCA2_Sahu_2023_exon13_Cisplatin_Resistance',
       'BRCA2_Sahu_2023_exon13_Olaparib_Resistance', 'BARD1_IGVF',
       'PALB2_IGVF', 'XRCC2_IGVF', 'SFPQ_IGVF', 'BRCA2_IGVF',
       'RAD51D_IGVF', 'CTCF_IGVF', 'BAP1_Waters_2024',
       'DDX3X_Radford_2023', 'RHO_Wan_2019', 'RAD51C_Olvera-León_2024',
       'BRCA2_Sahu_2025_SGE', 'BRCA2_Sahu_2023_exon13_global_score',
       'FKRP_Ma_2024', 'LARGE1_Ma_2024',
       'CARD11_Meitlis_2020_SGE_Ibrutinib_GoF',
       'CARD11_Meitlis_2020_SGE_LoF',
       'ASPA_Grønbæk-Thygesen_2024_abundance',
       'ASPA_Grønbæk-Thygesen_2024_toxicity',
       'BRCA1_Adamovich_2022_Cisplatin_Resistance',
       'BRCA1_Adamovich_2022_HDR', 'CALM1_CALM2_CALM3_Weile_2017',
       'CBS_Sun_2020_high_B6', 'CBS_Sun_2020_low_B6', 'CHEK2_Gebbia_2024',
       'CRX_Shepherdson_2024', 'F9_Popp_2025_carboxy_F9_specific',
       'F9_Popp_

In [7]:
dff.to_csv(
    OUT_DIR/"integrated_variant_effect_dataset.tsv.gz",
    sep="\t",
    index=False,
    compression="gzip"
)